Empiezo con una predicción lineal

In [2]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit # Para splits cronológicos múltiples
import time
import glob

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
N_LAGS = 24  # Aumentamos los lags, ya que predeciremos hasta 24h. Ajusta según EDA.
PREDICTION_HORIZONS = [1, 2, 4, 8, 12, 24] # Horizontes de predicción en horas
N_CV_SPLITS = 5 # Número de splits para TimeSeriesSplit (repeticiones de evaluación)
# TEST_SIZE_RATIO ya no se usa directamente, TimeSeriesSplit lo maneja.

# --- Rutas (Asegúrate de que coincidan con tu estructura) ---
PROJECT_ROOT_DIR = os.getcwd()

ORIGINAL_DATA_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs')
ORIGINAL_DATA_FILE = os.path.join(ORIGINAL_DATA_DIR, 'Air_Quality.csv')

SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')

INTERPOLATION_METHOD_NAME = 'interpolacionLineal'
IMPUTED_CSVS_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', INTERPOLATION_METHOD_NAME, 'CSVs')

PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'regresionLinealSimpleLags' # Mantener o cambiar si la estrategia es muy diferente
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)

CONTAMINANT_AND_AQI_COLS_IMPUTATION = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI', 'Date', 'City']


# --- Funciones Auxiliares ---

def create_lagged_features_for_horizon(df_series_input, target_col, n_lags, horizon):
    """
    Crea features de lags (hasta t-1) y el target y(t+h-1) -> y(t+horizon-1) para la estrategia directa.
    El índice de df_series_input se asume que es un RangeIndex (0 a N-1).
    """
    df_lagged = pd.DataFrame(index=df_series_input.index)
    
    # Crear Lags (features X)
    for i in range(1, n_lags + 1):
        df_lagged[f'{target_col}_lag{i}'] = df_series_input[target_col].shift(i)
        
    # Crear Target (y) para el horizonte especificado
    # y(t) será predicho usando X(t-1), X(t-2)...
    # Nuestro y real es AQI en el futuro. Si horizon=1, queremos predecir AQI(t) usando lags de AQI(t-1), AQI(t-2)...
    # El target y_t se alinea con los features X_t (que contienen lags del pasado de y_t)
    # Para predicción directa a h pasos: y_{t+h-1} = f(AQI_t-1, AQI_t-2, ...)
    # Pandas shift: shift(-k) adelanta los datos k periodos.
    # Queremos predecir `df_series_input[target_col]` en `horizon` pasos.
    # Entonces, el target es `df_series_input[target_col].shift(- (horizon -1) )` si los lags son de t.
    # Pero si nuestros lags son X_t = [y_{t-1}, y_{t-2}, ...], y queremos predecir y_{t+h-1},
    # el target es y.shift(-(horizon-1))
    # La forma más simple para estrategia directa: y es AQI.shift(-horizon) y los X son lags hasta el presente.
    
    df_lagged['target_h'] = df_series_input[target_col].shift(-horizon) # y_{t+h}
    
    # Eliminar NaNs generados por lags y por el shift del target
    df_lagged = df_lagged.dropna()
    
    if df_lagged.empty:
        return pd.DataFrame(), pd.Series(dtype='float64')

    y = df_lagged['target_h']
    X = df_lagged.drop(columns=['target_h'])
    
    return X, y


def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan # No se puede calcular diff
    d = np.abs(np.diff(y_train_for_naive)).sum() / (len(y_train_for_naive) - 1)
    if d == 0: return np.nan
    errors = np.abs(y_true - y_pred)
    return errors.mean() / d

def train_predict_eval_multi_horizon(df_city_data_input, target_col, n_lags, horizons_list, n_cv_splits):
    """
    Entrena, predice y evalúa para múltiples horizontes usando TimeSeriesSplit.
    df_city_data_input debe ser un DataFrame con una sola columna (la serie temporal target_col)
    y con índice reseteado (0 a N-1).
    """
    results_by_horizon = []

    if len(df_city_data_input) < n_lags + max(horizons_list) + n_cv_splits + 1:
        print(f"    Datos insuficientes para los horizontes y splits dados. Min requerido: {n_lags + max(horizons_list) + n_cv_splits +1}, disponible: {len(df_city_data_input)}")
        for h in horizons_list:
            results_by_horizon.append({'Horizon': h, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon

    for h_idx, horizon in enumerate(horizons_list):
        # print(f"      Horizonte: {horizon}h")
        X, y = create_lagged_features_for_horizon(df_city_data_input, target_col, n_lags, horizon)

        if X.empty or y.empty or len(X) < n_cv_splits * 2 : # Mínimo datos para TimeSeriesSplit
            # print(f"        Datos insuficientes tras crear lags/target para horizonte {horizon}h. X: {len(X)}, y: {len(y)}")
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue

        tscv = TimeSeriesSplit(n_splits=n_cv_splits)
        horizon_maes, horizon_rmses, horizon_mases = [], [], []

        for fold, (train_index, test_index) in enumerate(tscv.split(X)):
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]

            if X_train.empty or y_train.empty or X_test.empty or y_test.empty or len(y_train) <=1:
                # print(f"        Fold {fold+1}: Train o Test data vacíos o y_train muy corto. Omitiendo fold.")
                continue

            model = LinearRegression()
            try:
                model.fit(X_train, y_train)
                predictions = model.predict(X_test)

                mae = mean_absolute_error(y_test, predictions)
                rmse = np.sqrt(mean_squared_error(y_test, predictions))
                # Usar y_train de este fold para el MASE
                mase = calculate_mase(y_test.values, predictions, y_train.values)

                horizon_maes.append(mae)
                horizon_rmses.append(rmse)
                if not np.isnan(mase): horizon_mases.append(mase)
            except Exception as e:
                print(f"        Error en Fold {fold+1} para horizonte {horizon}h: {e}")
                continue
        
        results_by_horizon.append({
            'Horizon': horizon,
            'MAE': np.nanmean(horizon_maes) if horizon_maes else np.nan,
            'RMSE': np.nanmean(horizon_rmses) if horizon_rmses else np.nan,
            'MASE': np.nanmean(horizon_mases) if horizon_mases else np.nan
        })
    return results_by_horizon


# --- Script Principal ---
all_prediction_metrics_detailed = [] # Para guardar todas las repeticiones de CV y archivos
script_start_time = time.time()

# 1. Cargar DataFrame Original
try:
    df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    if 'CO2' in df_original_full.columns: df_original_full = df_original_full.drop(columns=['CO2'])
    print(f"DataFrame original '{ORIGINAL_DATA_FILE}' cargado.")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo original '{ORIGINAL_DATA_FILE}'. Omitiendo predicción con datos originales.")
    df_original_full = None

# Procesar DataFrame Original
if df_original_full is not None:
    print("\n--- Predicción con Datos Originales (sin NaNs simulados) ---")
    for city_name in df_original_full['City'].unique():
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        print(f"  Ciudad: {city_name}")
        # Trabajar con copia y resetear índice para consistencia con create_lagged_features_for_horizon
        df_city_original_series = df_original_full[df_original_full['City'] == city_name][[TARGET_VARIABLE]].copy().reset_index(drop=True)
        
        if df_city_original_series[TARGET_VARIABLE].isnull().any():
            # print(f"    Advertencia: NaNs en AQI para {city_name} (Original). Rellenando con ffill/bfill.")
            df_city_original_series[TARGET_VARIABLE] = df_city_original_series[TARGET_VARIABLE].ffill().bfill()
            if df_city_original_series[TARGET_VARIABLE].isnull().any():
                print(f"    No se pudieron rellenar NaNs en AQI para {city_name} (Original). Omitiendo.")
                continue
        
        horizon_metrics = train_predict_eval_multi_horizon(
            df_city_original_series, TARGET_VARIABLE, N_LAGS, PREDICTION_HORIZONS, N_CV_SPLITS
        )
        for hm in horizon_metrics:
            all_prediction_metrics_detailed.append({
                'Dataset_Type': 'Original', 'City': city_name, 'Scenario': 'N/A',
                'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'],
                'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']
            })
        # Imprimir resumen para el primer y último horizonte como ejemplo
        if horizon_metrics:
            print(f"    Métricas (Original, {horizon_metrics[0]['Horizon']}h): MAE={horizon_metrics[0]['MAE']:.4f}, RMSE={horizon_metrics[0]['RMSE']:.4f}, MASE={horizon_metrics[0]['MASE']:.4f}")
            if len(horizon_metrics) > 1:
                 print(f"    Métricas (Original, {horizon_metrics[-1]['Horizon']}h): MAE={horizon_metrics[-1]['MAE']:.4f}, RMSE={horizon_metrics[-1]['RMSE']:.4f}, MASE={horizon_metrics[-1]['MASE']:.4f}")


datasets_to_process_info = [
    ("Simulated_NaNs", SIMULATED_DATA_ROOT_DIR, "_with_nans.csv"),
    ("Imputed_Linear", IMPUTED_CSVS_ROOT_DIR, "_imputed_linear.csv")
]

for dataset_type, base_path, file_suffix in datasets_to_process_info:
    print(f"\n--- Predicción con Datos: {dataset_type} ---")
    if not os.path.exists(base_path):
        print(f"Directorio base no encontrado para {dataset_type}: {base_path}. Omitiendo.")
        continue

    for city_folder_name_safe in os.listdir(base_path):
        city_path = os.path.join(base_path, city_folder_name_safe)
        if not os.path.isdir(city_path): continue
        
        original_city_name = city_folder_name_safe.replace("_", " ") # Asumimos que el nombre de carpeta es el "safe"
        print(f"  Ciudad: {original_city_name} (desde carpeta: {city_folder_name_safe})")

        for scenario_folder_name_safe in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder_name_safe)
            if not os.path.isdir(scenario_path): continue
            print(f"    Escenario: {scenario_folder_name_safe}")

            pattern = os.path.join(scenario_path, f"*{file_suffix}")
            repetition_files = glob.glob(pattern)
            if not repetition_files: continue

            # Métricas para promediar sobre las repeticiones de este escenario/ciudad/horizonte
            # Estructura: {horizon: {'maes': [], 'rmses': [], 'mases': []}}
            scenario_horizon_metrics_accumulator = {h: {'maes': [], 'rmses': [], 'mases': []} for h in PREDICTION_HORIZONS}

            for rep_file_path in repetition_files:
                file_name = os.path.basename(rep_file_path)
                try:
                    # Extraer ID de repetición (ej. rep1_seed123)
                    rep_file_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                    df_rep = pd.read_csv(rep_file_path, usecols=lambda c: c in CONTAMINANT_AND_AQI_COLS_IMPUTATION)
                    df_rep['Date'] = pd.to_datetime(df_rep['Date'])
                except Exception as e:
                    print(f"        Error cargando/procesando archivo {rep_file_path}: {e}. Omitiendo.")
                    continue

                # Usar el nombre de ciudad del archivo si es diferente al de la carpeta (poco probable aquí)
                # current_city_in_file = df_rep['City'].unique()[0] if 'City' in df_rep.columns else original_city_name
                
                df_city_rep_series = df_rep[df_rep['City'] == original_city_name][[TARGET_VARIABLE]].copy().reset_index(drop=True)

                if dataset_type == "Simulated_NaNs" and df_city_rep_series[TARGET_VARIABLE].isnull().any():
                    df_city_rep_series[TARGET_VARIABLE] = df_city_rep_series[TARGET_VARIABLE].ffill().bfill()
                    if df_city_rep_series[TARGET_VARIABLE].isnull().any():
                        # print(f"        No se pudieron rellenar NaNs en {original_city_name} (Simulated_NaNs) para {file_name}. Omitiendo archivo.")
                        continue
                
                horizon_metrics_rep = train_predict_eval_multi_horizon(
                    df_city_rep_series, TARGET_VARIABLE, N_LAGS, PREDICTION_HORIZONS, N_CV_SPLITS
                )

                for hm_rep in horizon_metrics_rep:
                    h = hm_rep['Horizon']
                    # Guardar métrica detallada de esta repetición y split de CV (implícito en el promedio de hm_rep)
                    all_prediction_metrics_detailed.append({
                        'Dataset_Type': dataset_type, 'City': original_city_name, 'Scenario': scenario_folder_name_safe,
                        'Repetition_File_ID': rep_file_id, 'Horizon_h': h,
                        'MAE': hm_rep['MAE'], 'RMSE': hm_rep['RMSE'], 'MASE': hm_rep['MASE']
                    })
                    # Acumular para el promedio del escenario/horizonte
                    if not np.isnan(hm_rep['MAE']): scenario_horizon_metrics_accumulator[h]['maes'].append(hm_rep['MAE'])
                    if not np.isnan(hm_rep['RMSE']): scenario_horizon_metrics_accumulator[h]['rmses'].append(hm_rep['RMSE'])
                    if not np.isnan(hm_rep['MASE']): scenario_horizon_metrics_accumulator[h]['mases'].append(hm_rep['MASE'])
            
            # Calcular y mostrar promedios para el escenario y algunos horizontes
            print(f"      Promedios para Escenario '{scenario_folder_name_safe}':")
            for h_summary in [PREDICTION_HORIZONS[0], PREDICTION_HORIZONS[-1]]: # Mostrar primero y último horizonte
                 if h_summary in scenario_horizon_metrics_accumulator:
                    avg_s_mae = np.nanmean(scenario_horizon_metrics_accumulator[h_summary]['maes'])
                    avg_s_rmse = np.nanmean(scenario_horizon_metrics_accumulator[h_summary]['rmses'])
                    avg_s_mase = np.nanmean(scenario_horizon_metrics_accumulator[h_summary]['mases'])
                    print(f"        Horizonte {h_summary}h: MAE={avg_s_mae:.4f}, RMSE={avg_s_rmse:.4f}, MASE={avg_s_mase:.4f}")


# --- Guardado de Métricas ---
df_all_prediction_metrics_detailed = pd.DataFrame(all_prediction_metrics_detailed)

# 1. Guardar el CSV detallado global
if not df_all_prediction_metrics_detailed.empty:
    detailed_output_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_prediction_metrics_detailed.to_csv(detailed_output_path, index=False)
    print(f"\nTodas las métricas de predicción detalladas guardadas en: {detailed_output_path}")
    print(df_all_prediction_metrics_detailed.head())

# 2. Crear y guardar CSV de métricas promedio por ciudad
# Agrupar por Ciudad, Dataset_Type, Scenario, Horizon_h y calcular el promedio de MAE, RMSE, MASE
if not df_all_prediction_metrics_detailed.empty:
    df_summary_by_city_scenario_horizon = df_all_prediction_metrics_detailed.groupby(
        ['City', 'Dataset_Type', 'Scenario', 'Horizon_h']
    ).agg(
        Avg_MAE=('MAE', 'mean'),
        Avg_RMSE=('RMSE', 'mean'),
        Avg_MASE=('MASE', 'mean'),
        Std_MAE=('MAE', 'std'), # Opcional: Desviación estándar
        Std_RMSE=('RMSE', 'std'),
        Std_MASE=('MASE', 'std'),
        Count_Repetitions=('MAE', 'count') # Para verificar cuántas repeticiones contribuyeron
    ).reset_index()

    for city_name_original_unique in df_summary_by_city_scenario_horizon['City'].unique():
        city_name_s = city_name_original_unique.replace(" ", "_").replace("-", "_")
        df_city_summary = df_summary_by_city_scenario_horizon[df_summary_by_city_scenario_horizon['City'] == city_name_original_unique]
        
        city_metrics_dir = os.path.join(PREDICTION_METRICS_ROOT_DIR, city_name_s) # Nueva subcarpeta por ciudad
        os.makedirs(city_metrics_dir, exist_ok=True)
        city_summary_file_path = os.path.join(city_metrics_dir, f'metrics_summary_{PREDICTION_METHOD_NAME}_{city_name_s}.csv')
        
        df_city_summary.to_csv(city_summary_file_path, index=False)
        print(f"  Resumen de métricas para {city_name_original_unique} guardado en: {city_summary_file_path}")

# 3. Crear y guardar CSV de métricas globales promediadas por Dataset_Type y Horizon_h (sobre todas las ciudades y escenarios)
if not df_all_prediction_metrics_detailed.empty:
    df_global_summary_by_dataset_horizon = df_all_prediction_metrics_detailed.groupby(
        ['Dataset_Type', 'Horizon_h'] # Podrías añadir 'Scenario' aquí si quieres ver el promedio por escenario globalmente
    ).agg(
        Global_Avg_MAE=('MAE', 'mean'),
        Global_Avg_RMSE=('RMSE', 'mean'),
        Global_Avg_MASE=('MASE', 'mean')
    ).reset_index()
    
    global_summary_file_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_global_summary_{PREDICTION_METHOD_NAME}.csv')
    df_global_summary_by_dataset_horizon.to_csv(global_summary_file_path, index=False)
    print(f"\nResumen global de métricas promedio por tipo de dataset y horizonte guardado en: {global_summary_file_path}")
    print(df_global_summary_by_dataset_horizon)


script_end_time = time.time()
print(f"\nScript de predicción completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")

DataFrame original 'c:\Users\fdemiguel\Documentos\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' cargado.

--- Predicción con Datos Originales (sin NaNs simulados) ---
  Ciudad: Brasilia
    Métricas (Original, 1h): MAE=1.8579, RMSE=2.6920, MASE=1.1286
    Métricas (Original, 24h): MAE=3.8425, RMSE=5.3649, MASE=2.3295
  Ciudad: Cairo
    Métricas (Original, 1h): MAE=1.9158, RMSE=3.2011, MASE=1.9336
    Métricas (Original, 24h): MAE=9.7798, RMSE=12.2245, MASE=10.1662
  Ciudad: Dubai
    Métricas (Original, 1h): MAE=0.8385, RMSE=1.5415, MASE=0.9591
    Métricas (Original, 24h): MAE=13.0547, RMSE=17.4330, MASE=14.8770
  Ciudad: London
    Métricas (Original, 1h): MAE=2.0932, RMSE=2.8293, MASE=1.7445
    Métricas (Original, 24h): MAE=6.0932, RMSE=8.3926, MASE=5.0437
  Ciudad: New York
    Métricas (Original, 1h): MAE=2.7261, RMSE=4.0730, MASE=1.6947
    Métricas (Original, 24h): MAE=9.1634, RMSE=11.8341, MASE=5.6096
  Ciudad: Sydney
    Métricas (Original, 1h): MAE=1.8503, RMSE=2.8655, MA

SARIMA

In [4]:
import pandas as pd
import numpy as np
import os
import json
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit # Para validación cruzada cronológica
import time
import glob
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning) # Para statsmodels

# --- 0. Configuración Inicial y Parámetros ---
TARGET_CITY_SARIMA = "Brasilia" # <--- MODIFICACIÓN: Enfocarse en Brasilia
TARGET_VARIABLE_SARIMA = 'AQI'
PREDICTION_HORIZONS_SARIMA = [1, 2, 4, 8, 12, 24]
N_CV_SPLITS_SARIMA = 2
COLUMNS_TO_LOAD_FROM_CSV = ['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']

# --- Rutas ---
PROJECT_ROOT_DIR_SARIMA_PRED = os.getcwd()

ORIGINAL_DATA_DIR_SARIMA = os.path.join(PROJECT_ROOT_DIR_SARIMA_PRED, 'CSVs')
ORIGINAL_DATA_FILE_SARIMA = os.path.join(ORIGINAL_DATA_DIR_SARIMA, 'Air_Quality.csv')

# SIMULATED_DATA_ROOT_DIR_SARIMA = os.path.join(PROJECT_ROOT_DIR_SARIMA_PRED, 'CSVs', 'simulados') # No se usa directamente para cargar, sino como referencia de estructura

IMPUTATIONS_BASE_INPUT_DIR_SARIMA = os.path.join(PROJECT_ROOT_DIR_SARIMA_PRED, 'imputaciones')
ORDERS_JSON_INPUT_DIR_SARIMA = os.path.join(PROJECT_ROOT_DIR_SARIMA_PRED, 'imputaciones', 'SARIMA', 'model_orders')

PREDICTION_METHOD_NAME_SARIMA = 'SARIMA_pred'
PREDICTION_METRICS_SARIMA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_PRED, 'predicciones', PREDICTION_METHOD_NAME_SARIMA, 'metricas')
os.makedirs(PREDICTION_METRICS_SARIMA_ROOT_DIR, exist_ok=True)


# --- Funciones Auxiliares (sin cambios lógicos internos) ---
def calculate_mase_pred(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    mae_naive_on_train = np.mean(np.abs(np.diff(y_train_for_naive)))
    if mae_naive_on_train == 0 or np.isnan(mae_naive_on_train): return np.nan
    actual_forecast_errors = np.abs(y_true - y_pred)
    mase = np.mean(actual_forecast_errors) / mae_naive_on_train
    return mase

def train_predict_eval_sarima_multi_horizon(
    df_series_target_input, sarima_order, sarima_seasonal_order,
    horizons_list, n_cv_splits
):
    results_by_horizon_sarima = []
    series_data_for_sarima = df_series_target_input.iloc[:, 0]
    min_obs_needed = max(sum(sarima_order[:2]), sum(sarima_seasonal_order[:2])) + sarima_seasonal_order[3] if sarima_seasonal_order[3] > 0 else max(sum(sarima_order[:2]),1)
    min_obs_needed = max(min_obs_needed * (n_cv_splits +1) , n_cv_splits * 20, max(horizons_list) + 50) # Aumentado umbral
    
    if len(series_data_for_sarima) < min_obs_needed:
        for h_s in horizons_list:
            results_by_horizon_sarima.append({'Horizon': h_s, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon_sarima

    tscv_sarima = TimeSeriesSplit(n_splits=n_cv_splits)
    accumulated_metrics_all_horizons = {h: {'maes': [], 'rmses': [], 'mases': []} for h in horizons_list}

    for fold, (train_idx_pos, test_idx_pos) in enumerate(tscv_sarima.split(series_data_for_sarima)):
        train_series = series_data_for_sarima.iloc[train_idx_pos]
        test_series = series_data_for_sarima.iloc[test_idx_pos]

        # Aumentar umbral para train_series en cada fold también
        if len(train_series) < (min_obs_needed / (n_cv_splits + 1)) or len(train_series) <= max(sum(sarima_order[:2]), sum(sarima_seasonal_order[:2])) + 10 :
            continue
        if len(test_series) == 0 : continue # Evitar error si el test set es vacío

        try:
            model_sarima = SARIMAX(train_series, order=sarima_order, seasonal_order=sarima_seasonal_order,
                                   enforce_stationarity=False, enforce_invertibility=False,
                                   initialization='approximate_diffuse')
            results_sarima = model_sarima.fit(disp=False, maxiter=150)
            max_steps_to_forecast_for_fold = len(test_series)
            if max_steps_to_forecast_for_fold > 0:
                forecasts_all_steps = results_sarima.forecast(steps=max_steps_to_forecast_for_fold)
            else: continue # No hay nada que predecir

            for h_s in horizons_list:
                if h_s <= max_steps_to_forecast_for_fold:
                    y_true_h = test_series.iloc[:h_s]
                    y_pred_h = forecasts_all_steps.iloc[:h_s]
                    common_len = min(len(y_true_h), len(y_pred_h))
                    if common_len == 0: continue
                    y_true_h_common = y_true_h.iloc[:common_len].values
                    y_pred_h_common = y_pred_h.iloc[:common_len].values
                    
                    if len(y_true_h_common) > 0: # Asegurar que haya datos para calcular métricas
                        mae_h = mean_absolute_error(y_true_h_common, y_pred_h_common)
                        rmse_h = np.sqrt(mean_squared_error(y_true_h_common, y_pred_h_common))
                        mase_h = calculate_mase_pred(y_true_h_common, y_pred_h_common, train_series.values)
                        accumulated_metrics_all_horizons[h_s]['maes'].append(mae_h)
                        accumulated_metrics_all_horizons[h_s]['rmses'].append(rmse_h)
                        if not np.isnan(mase_h): accumulated_metrics_all_horizons[h_s]['mases'].append(mase_h)
        except Exception: continue
            
    for h_s in horizons_list:
        results_by_horizon_sarima.append({
            'Horizon': h_s,
            'MAE': np.nanmean(accumulated_metrics_all_horizons[h_s]['maes']) if accumulated_metrics_all_horizons[h_s]['maes'] else np.nan,
            'RMSE': np.nanmean(accumulated_metrics_all_horizons[h_s]['rmses']) if accumulated_metrics_all_horizons[h_s]['rmses'] else np.nan,
            'MASE': np.nanmean(accumulated_metrics_all_horizons[h_s]['mases']) if accumulated_metrics_all_horizons[h_s]['mases'] else np.nan
        })
    return results_by_horizon_sarima

# --- Script Principal de Predicción SARIMA ---
all_prediction_metrics_sarima_detailed = []
script_start_time_sarima_pred = time.time()

try:
    df_original_full_sarima = pd.read_csv(ORIGINAL_DATA_FILE_SARIMA)
    df_original_full_sarima['Date'] = pd.to_datetime(df_original_full_sarima['Date'])
    if 'CO2' in df_original_full_sarima.columns: df_original_full_sarima = df_original_full_sarima.drop(columns=['CO2'])
    print(f"DataFrame original '{ORIGINAL_DATA_FILE_SARIMA}' cargado para predicción SARIMA.")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo original '{ORIGINAL_DATA_FILE_SARIMA}'.")
    df_original_full_sarima = None

datasets_to_predict_with_sarima = [
    ("Original_Clean", ORIGINAL_DATA_DIR_SARIMA, ".csv"),
    ("Imputed_LinearInterpolation", os.path.join(IMPUTATIONS_BASE_INPUT_DIR_SARIMA, 'interpolacionLineal', 'CSVs'), "_imputed_linear.csv"),
    ("Imputed_SARIMA", os.path.join(IMPUTATIONS_BASE_INPUT_DIR_SARIMA, 'SARIMA', 'CSVs'), "_imputed_sarima.csv"),
]

for dataset_type_sarima, base_path_sarima, file_suffix_sarima in datasets_to_predict_with_sarima:
    print(f"\n--- Predicción SARIMA con Datos: {dataset_type_sarima} ---")
    if not os.path.exists(base_path_sarima):
        print(f"Directorio base no encontrado para {dataset_type_sarima}: {base_path_sarima}. Omitiendo.")
        continue

    if dataset_type_sarima == "Original_Clean":
        if df_original_full_sarima is None: continue
        
        # MODIFICACIÓN: Procesar solo TARGET_CITY_SARIMA
        if TARGET_CITY_SARIMA not in df_original_full_sarima['City'].unique():
            print(f"  Ciudad {TARGET_CITY_SARIMA} no encontrada en datos originales. Omitiendo Original_Clean para esta ciudad.")
            continue

        city_name_orig_s = TARGET_CITY_SARIMA
        city_name_safe_s = city_name_orig_s.replace(" ", "_").replace("-", "_")
        print(f"  Ciudad: {city_name_orig_s} (Dataset: {dataset_type_sarima})")
        
        df_city_data_orig_s = df_original_full_sarima[df_original_full_sarima['City'] == city_name_orig_s][[TARGET_VARIABLE_SARIMA]].copy().reset_index(drop=True)
        if df_city_data_orig_s[TARGET_VARIABLE_SARIMA].isnull().any():
            df_city_data_orig_s[TARGET_VARIABLE_SARIMA] = df_city_data_orig_s[TARGET_VARIABLE_SARIMA].ffill().bfill()
        if df_city_data_orig_s[TARGET_VARIABLE_SARIMA].isnull().any():
            print(f"    No se pudieron rellenar NaNs en {TARGET_VARIABLE_SARIMA} para {city_name_orig_s} (Original). Omitiendo.")
            continue

        order_file_path_s = os.path.join(ORDERS_JSON_INPUT_DIR_SARIMA, f'orders_{city_name_safe_s}_{TARGET_VARIABLE_SARIMA}.json')
        try:
            with open(order_file_path_s, 'r') as f: orders_s = json.load(f)
            order_tuple_s = tuple(orders_s['order']); seasonal_order_tuple_s = tuple(orders_s['seasonal_order'])
        except Exception: # Simplificado el manejo de error de JSON
            print(f"    Órdenes no encontradas o error en {order_file_path_s}. Usando (1,0,1)(0,0,0,0).")
            order_tuple_s = (1,0,1); seasonal_order_tuple_s = (0,0,0,0)

        horizon_metrics_s = train_predict_eval_sarima_multi_horizon(
            df_city_data_orig_s, order_tuple_s, seasonal_order_tuple_s,
            PREDICTION_HORIZONS_SARIMA, N_CV_SPLITS_SARIMA
        )
        for hm_s in horizon_metrics_s:
            all_prediction_metrics_sarima_detailed.append({
                'Dataset_Type': dataset_type_sarima, 'City': city_name_orig_s, 'Scenario': 'N/A_Original',
                'Repetition_File_ID': 'N/A_Original', 'Horizon_h': hm_s['Horizon'],
                'MAE': hm_s['MAE'], 'RMSE': hm_s['RMSE'], 'MASE': hm_s['MASE']
            })
        if horizon_metrics_s and horizon_metrics_s[0]['MAE'] is not np.nan : # Imprimir si hay métricas válidas
            print(f"    Métricas SARIMA ({dataset_type_sarima}, H={horizon_metrics_s[0]['Horizon']}h): MAE={horizon_metrics_s[0]['MAE']:.4f}, RMSE={horizon_metrics_s[0]['RMSE']:.4f}, MASE={horizon_metrics_s[0]['MASE']:.4f}")
            if len(horizon_metrics_s) > 1 and horizon_metrics_s[-1]['MAE'] is not np.nan:
                print(f"    Métricas SARIMA ({dataset_type_sarima}, H={horizon_metrics_s[-1]['Horizon']}h): MAE={horizon_metrics_s[-1]['MAE']:.4f}, RMSE={horizon_metrics_s[-1]['RMSE']:.4f}, MASE={horizon_metrics_s[-1]['MASE']:.4f}")
        continue

    # Para datasets imputados
    city_folder_name_safe_target = TARGET_CITY_SARIMA.replace(" ", "_").replace("-", "_")
    city_path_s = os.path.join(base_path_sarima, city_folder_name_safe_target) # Ruta directa a la carpeta de Brasilia

    if not os.path.isdir(city_path_s):
        print(f"  Carpeta de la ciudad {TARGET_CITY_SARIMA} no encontrada en {base_path_sarima}. Omitiendo para {dataset_type_sarima}.")
        continue
        
    original_city_name_s = TARGET_CITY_SARIMA # Usar el nombre original para logs y almacenamiento de métricas
    print(f"  Ciudad: {original_city_name_s} (Dataset: {dataset_type_sarima}, desde carpeta: {city_folder_name_safe_target})")

    order_file_path_s = os.path.join(ORDERS_JSON_INPUT_DIR_SARIMA, f'orders_{city_folder_name_safe_target}_{TARGET_VARIABLE_SARIMA}.json')
    try:
        with open(order_file_path_s, 'r') as f: orders_s = json.load(f)
        order_tuple_s = tuple(orders_s['order']); seasonal_order_tuple_s = tuple(orders_s['seasonal_order'])
    except Exception: # Simplificado
        print(f"    Órdenes no encontradas o error para {TARGET_VARIABLE_SARIMA} en {order_file_path_s}. Usando (1,0,1)(0,0,0,0).")
        order_tuple_s = (1,0,1); seasonal_order_tuple_s = (0,0,0,0) # Fallback, aunque idealmente debería detenerse si es crítico

    for scenario_folder_name_safe_s in os.listdir(city_path_s):
        scenario_path_s = os.path.join(city_path_s, scenario_folder_name_safe_s)
        if not os.path.isdir(scenario_path_s): continue
        print(f"    Escenario: {scenario_folder_name_safe_s}")

        pattern_s = os.path.join(scenario_path_s, f"*{file_suffix_sarima}")
        repetition_files_s = sorted(glob.glob(pattern_s))
        if not repetition_files_s: continue

        scenario_horizon_metrics_accumulator_s = {h: {'maes': [], 'rmses': [], 'mases': []} for h in PREDICTION_HORIZONS_SARIMA}

        for rep_file_path_s in repetition_files_s:
            file_name_s = os.path.basename(rep_file_path_s)
            try:
                rep_file_id_s = "_".join(file_name_s.split('_')[-4:-2]) if "_rep" in file_name_s and "_seed" in file_name_s else file_name_s
                df_rep_s = pd.read_csv(rep_file_path_s, usecols=lambda c: c in COLUMNS_TO_LOAD_FROM_CSV)
            except Exception as e_load_s:
                print(f"        Error cargando/procesando archivo {rep_file_path_s}: {e_load_s}. Omitiendo.")
                continue
            
            df_city_rep_target_series_s = df_rep_s[df_rep_s['City'] == original_city_name_s][[TARGET_VARIABLE_SARIMA]].copy().reset_index(drop=True)
            if df_city_rep_target_series_s[TARGET_VARIABLE_SARIMA].isnull().any():
                print(f"        Advertencia: NaNs encontrados en {TARGET_VARIABLE_SARIMA} del archivo imputado {file_name_s}. Rellenando.")
                df_city_rep_target_series_s[TARGET_VARIABLE_SARIMA] = df_city_rep_target_series_s[TARGET_VARIABLE_SARIMA].ffill().bfill()
                if df_city_rep_target_series_s[TARGET_VARIABLE_SARIMA].isnull().any():
                     print(f"        No se pudieron rellenar NaNs en {TARGET_VARIABLE_SARIMA} para {file_name_s}. Omitiendo archivo.")
                     continue
            
            horizon_metrics_rep_s = train_predict_eval_sarima_multi_horizon(
                df_city_rep_target_series_s, order_tuple_s, seasonal_order_tuple_s,
                PREDICTION_HORIZONS_SARIMA, N_CV_SPLITS_SARIMA
            )
            for hm_rep_s in horizon_metrics_rep_s:
                h_s_val = hm_rep_s['Horizon']
                all_prediction_metrics_sarima_detailed.append({
                    'Dataset_Type': dataset_type_sarima, 'City': original_city_name_s, 'Scenario': scenario_folder_name_safe_s,
                    'Repetition_File_ID': rep_file_id_s, 'Horizon_h': h_s_val,
                    'MAE': hm_rep_s['MAE'], 'RMSE': hm_rep_s['RMSE'], 'MASE': hm_rep_s['MASE']
                })
                if not np.isnan(hm_rep_s['MAE']): scenario_horizon_metrics_accumulator_s[h_s_val]['maes'].append(hm_rep_s['MAE'])
                if not np.isnan(hm_rep_s['RMSE']): scenario_horizon_metrics_accumulator_s[h_s_val]['rmses'].append(hm_rep_s['RMSE'])
                if not np.isnan(hm_rep_s['MASE']): scenario_horizon_metrics_accumulator_s[h_s_val]['mases'].append(hm_rep_s['MASE'])
        
        if any(scenario_horizon_metrics_accumulator_s[h]['maes'] for h in scenario_horizon_metrics_accumulator_s): # Si hay alguna métrica
            print(f"      Promedios SARIMA para Escenario '{scenario_folder_name_safe_s}':")
            for h_summary_s in [PREDICTION_HORIZONS_SARIMA[0], PREDICTION_HORIZONS_SARIMA[-1]]:
                 if h_summary_s in scenario_horizon_metrics_accumulator_s and scenario_horizon_metrics_accumulator_s[h_summary_s]['maes']:
                    avg_s_mae_s = np.nanmean(scenario_horizon_metrics_accumulator_s[h_summary_s]['maes'])
                    avg_s_rmse_s = np.nanmean(scenario_horizon_metrics_accumulator_s[h_summary_s]['rmses'])
                    avg_s_mase_s = np.nanmean(scenario_horizon_metrics_accumulator_s[h_summary_s]['mases'])
                    print(f"        Horizonte {h_summary_s}h: MAE={avg_s_mae_s:.4f}, RMSE={avg_s_rmse_s:.4f}, MASE={avg_s_mase_s:.4f}")

# --- Guardado de Métricas de Predicción SARIMA ---
df_all_prediction_metrics_sarima_detailed = pd.DataFrame(all_prediction_metrics_sarima_detailed)

if not df_all_prediction_metrics_sarima_detailed.empty:
    # Guardar métricas detalladas (ahora solo de Brasilia)
    detailed_output_path_s = os.path.join(PREDICTION_METRICS_SARIMA_ROOT_DIR, f'Brasilia_prediction_metrics_detailed_{PREDICTION_METHOD_NAME_SARIMA}.csv')
    df_all_prediction_metrics_sarima_detailed.to_csv(detailed_output_path_s, index=False)
    print(f"\nMétricas de predicción SARIMA detalladas (para {TARGET_CITY_SARIMA}) guardadas en: {detailed_output_path_s}")
    print(df_all_prediction_metrics_sarima_detailed.head())

    # El resumen por ciudad ahora será solo de Brasilia.
    # La carpeta para las métricas de ciudad se llamará "Brasilia"
    city_name_s_file_summary = TARGET_CITY_SARIMA.replace(" ", "_").replace("-", "_")
    city_metrics_dir_s_summary = os.path.join(PREDICTION_METRICS_SARIMA_ROOT_DIR, city_name_s_file_summary) # Ej: predicciones/SARIMA_pred/metricas/Brasilia/
    os.makedirs(city_metrics_dir_s_summary, exist_ok=True)
    
    df_summary_for_target_city = df_all_prediction_metrics_sarima_detailed[df_all_prediction_metrics_sarima_detailed['City'] == TARGET_CITY_SARIMA]
    
    if not df_summary_for_target_city.empty:
        df_summary_by_dataset_scenario_horizon_s = df_summary_for_target_city.groupby(
            ['Dataset_Type', 'Scenario', 'Horizon_h'] # City ya es fija
        ).agg(
            Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean'),
            Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Std_MASE=('MASE', 'std'),
            Count_Repetitions=('MAE', 'count')
        ).reset_index()

        city_summary_file_path_s = os.path.join(city_metrics_dir_s_summary, f'metrics_summary_{PREDICTION_METHOD_NAME_SARIMA}_{city_name_s_file_summary}.csv')
        df_summary_by_dataset_scenario_horizon_s.to_csv(city_summary_file_path_s, index=False)
        print(f"  Resumen de métricas SARIMA para {TARGET_CITY_SARIMA} guardado en: {city_summary_file_path_s}")

    # El resumen "global" también será solo de Brasilia
    df_global_summary_by_dataset_horizon_s = df_all_prediction_metrics_sarima_detailed.groupby(
        ['Dataset_Type', 'Horizon_h'] # Agrupar sin ciudad, ya que solo hay una
    ).agg(
        Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')
    ).reset_index()
    
    global_summary_file_path_s = os.path.join(PREDICTION_METRICS_SARIMA_ROOT_DIR, f'Brasilia_prediction_metrics_global_summary_{PREDICTION_METHOD_NAME_SARIMA}.csv')
    df_global_summary_by_dataset_horizon_s.to_csv(global_summary_file_path_s, index=False)
    print(f"\nResumen 'global' (para {TARGET_CITY_SARIMA}) de métricas SARIMA promedio guardado en: {global_summary_file_path_s}")
    if not df_global_summary_by_dataset_horizon_s.empty: print(df_global_summary_by_dataset_horizon_s)

script_end_time_sarima_pred = time.time()
print(f"\nScript de predicción SARIMA completado. Tiempo total: {(script_end_time_sarima_pred - script_start_time_sarima_pred)/60:.2f} minutos.")

DataFrame original 'c:\Users\fdemiguel\Documentos\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' cargado para predicción SARIMA.

--- Predicción SARIMA con Datos: Original_Clean ---
  Ciudad: Brasilia (Dataset: Original_Clean)
    Métricas SARIMA (Original_Clean, H=1h): MAE=0.9019, RMSE=0.9019, MASE=0.5634
    Métricas SARIMA (Original_Clean, H=24h): MAE=1.2857, RMSE=1.5836, MASE=0.7954

--- Predicción SARIMA con Datos: Imputed_LinearInterpolation ---
  Ciudad: Brasilia (Dataset: Imputed_LinearInterpolation, desde carpeta: Brasilia)
    Escenario: BlockRows_12h
      Promedios SARIMA para Escenario 'BlockRows_12h':
        Horizonte 1h: MAE=0.8958, RMSE=0.8958, MASE=0.5641
        Horizonte 24h: MAE=1.2611, RMSE=1.5568, MASE=0.7866
    Escenario: BlockRows_24h
      Promedios SARIMA para Escenario 'BlockRows_24h':
        Horizonte 1h: MAE=1.0757, RMSE=1.0757, MASE=0.6727
        Horizonte 24h: MAE=1.5533, RMSE=1.8826, MASE=0.9633
    Escenario: BlockRows_2h
      Promedios SARIMA par

RF

In [7]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Configuración Inicial y Parámetros ---
TARGET_CITY_RF_PRED = "Brasilia"
TARGET_VARIABLE_RF_PRED = 'AQI'
CONTAMINANT_COLS_FOR_FEATURES = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10'] # Exógenas

# Lags para las features del modelo de predicción RF
N_LAGS_AQI = 24  # Lags para la variable objetivo (AQI)
N_LAGS_EXO = 3   # Lags para las variables exógenas (contaminantes). Poner a 0 si solo queremos contemporáneas.

PREDICTION_HORIZONS_RF_PRED = [1, 2, 4, 8, 12, 24]
N_CV_SPLITS_RF_PRED = 3 # Reducido para pruebas más rápidas, puedes aumentar a 5

TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

# Columnas a cargar de los CSVs: Date, City, AQI, todos los CONTAMINANT_COLS_FOR_FEATURES, y TIME_FEATURES
COLUMNS_TO_LOAD_RF_PRED = ['Date', 'City', TARGET_VARIABLE_RF_PRED] + CONTAMINANT_COLS_FOR_FEATURES + TIME_FEATURES
# Eliminar duplicados si TARGET_VARIABLE_RF_PRED está en CONTAMINANT_COLS_FOR_FEATURES (no debería)
COLUMNS_TO_LOAD_RF_PRED = sorted(list(set(COLUMNS_TO_LOAD_RF_PRED)))


# --- Rutas ---
PROJECT_ROOT_DIR_RF_PRED = os.getcwd()
ORIGINAL_DATA_DIR_RF_PRED = os.path.join(PROJECT_ROOT_DIR_RF_PRED, 'CSVs')
ORIGINAL_DATA_FILE_RF_PRED = os.path.join(ORIGINAL_DATA_DIR_RF_PRED, 'Air_Quality.csv')
IMPUTATIONS_BASE_INPUT_DIR_RF_PRED = os.path.join(PROJECT_ROOT_DIR_RF_PRED, 'imputaciones')
IMPUTED_RF_CSVS_ROOT_DIR = os.path.join(IMPUTATIONS_BASE_INPUT_DIR_RF_PRED, 'RandomForestImputer', 'CSVs')
SIMULATED_NANS_CSVS_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_RF_PRED, 'CSVs', 'simulados')

PREDICTION_METHOD_NAME_RF_PRED = 'RandomForestPred' # Nuevo nombre para diferenciar
PREDICTION_METRICS_RF_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_RF_PRED, 'predicciones', PREDICTION_METHOD_NAME_RF_PRED, 'metricas')
os.makedirs(PREDICTION_METRICS_RF_ROOT_DIR, exist_ok=True)


# --- Funciones Auxiliares ---
def create_multivariate_lagged_features_rf(df_full_city_data, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizon):
    """
    Crea features de lags para la variable objetivo y variables exógenas, más features de tiempo.
    El target es y_{t+horizon}. Las features son de tiempo t y anteriores.
    """
    df_processed = pd.DataFrame(index=df_full_city_data.index)

    # Lags de la variable objetivo (ej. AQI_t-1, AQI_t-2, ...)
    for i in range(1, n_lags_target + 1):
        df_processed[f'{target_col}_lag{i}'] = df_full_city_data[target_col].shift(i)

    # Valores contemporáneos (t) y lags de variables exógenas
    for exo_col in exo_cols:
        if exo_col == target_col: continue # Ya se manejaron los lags del target
        # Valor contemporáneo (t) de la exógena. El imputer ya debería haberlo rellenado.
        df_processed[f'{exo_col}_t0'] = df_full_city_data[exo_col] #.shift(0) es redundante
        if n_lags_exo > 0:
            for i in range(1, n_lags_exo + 1):
                df_processed[f'{exo_col}_lag{i}'] = df_full_city_data[exo_col].shift(i)
    
    # Features de tiempo (contemporáneas, t)
    for tf_col in time_features_cols:
        if tf_col in df_full_city_data.columns:
            df_processed[tf_col] = df_full_city_data[tf_col]
            
    # Target para predicción directa: y_{t+horizon}
    # (shift(-horizon) adelanta los datos, por lo que el valor en la fila t se convierte en y_{t+horizon})
    df_processed['target_h'] = df_full_city_data[target_col].shift(-horizon)
    
    df_processed = df_processed.dropna() # Eliminar filas con cualquier NaN generado por los shifts
    
    if df_processed.empty: return pd.DataFrame(), pd.Series(dtype='float64')
    
    y = df_processed['target_h']
    X = df_processed.drop(columns=['target_h'])
    return X, y

def calculate_mase_pred_rf(y_true, y_pred, y_train_for_naive): # Sin cambios
    if len(y_train_for_naive) <= 1: return np.nan
    train_vals = np.asarray(y_train_for_naive)
    if len(train_vals) <= 1: return np.nan
    mae_naive_on_train = np.mean(np.abs(np.diff(train_vals)))
    if mae_naive_on_train == 0 or np.isnan(mae_naive_on_train) or mae_naive_on_train < 1e-9 : return np.nan
    true_vals, pred_vals = np.asarray(y_true), np.asarray(y_pred)
    actual_forecast_errors = np.abs(true_vals - pred_vals)
    mase = np.mean(actual_forecast_errors) / mae_naive_on_train
    return mase

def train_predict_eval_rf_multi_horizon(
    df_city_data_input_rf, # DataFrame con TARGET_VARIABLE, EXO_COLS y TIME_FEATURES
    target_col_rf, exo_cols_rf, time_features_list_rf, 
    n_lags_target_rf, n_lags_exo_rf,
    horizons_list_rf, n_cv_splits_rf
):
    results_by_horizon_rf = []
    # Heurística para longitud mínima de datos
    min_total_len = max(n_lags_target_rf, n_lags_exo_rf if n_lags_exo_rf > 0 else 0) + max(horizons_list_rf) + n_cv_splits_rf * 10
    
    if len(df_city_data_input_rf) < min_total_len:
        for h_rf in horizons_list_rf:
            results_by_horizon_rf.append({'Horizon': h_rf, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon_rf

    for horizon_rf in horizons_list_rf:
        X_rf, y_rf = create_multivariate_lagged_features_rf(
            df_city_data_input_rf, target_col_rf, exo_cols_rf, time_features_list_rf,
            n_lags_target_rf, n_lags_exo_rf, horizon_rf
        )
        if X_rf.empty or y_rf.empty or len(X_rf) < n_cv_splits_rf * 2:
            results_by_horizon_rf.append({'Horizon': horizon_rf, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue

        tscv_rf = TimeSeriesSplit(n_splits=n_cv_splits_rf)
        horizon_maes_rf, horizon_rmses_rf, horizon_mases_rf = [], [], []

        for fold_rf, (train_idx_rf, test_idx_rf) in enumerate(tscv_rf.split(X_rf)):
            X_train_rf, X_test_rf = X_rf.iloc[train_idx_rf], X_rf.iloc[test_idx_rf]
            y_train_rf, y_test_rf = y_rf.iloc[train_idx_rf], y_rf.iloc[test_idx_rf]

            if X_train_rf.empty or y_train_rf.empty or X_test_rf.empty or y_test_rf.empty or len(y_train_rf) <=1:
                continue
            
            # --- AJUSTE DE HIPERPARÁMETROS RF ---
            model_rf = RandomForestRegressor(
                n_estimators=150,       # Aumentado
                random_state=42,
                n_jobs=-1,
                max_depth=None,        # Permitir que los árboles crezcan (regular con min_samples)
                min_samples_split=5,   # Ajustar
                min_samples_leaf=3,    # Ajustar
                max_features='sqrt',   # Común para RF
                bootstrap=True         # Default, bueno para RF
            )
            try:
                model_rf.fit(X_train_rf, y_train_rf)
                predictions_rf = model_rf.predict(X_test_rf)
                mae_rf = mean_absolute_error(y_test_rf, predictions_rf)
                rmse_rf = np.sqrt(mean_squared_error(y_test_rf, predictions_rf))
                
                # Para MASE: y_train_for_naive es la serie original del AQI del fold de entrenamiento
                original_series_indices_for_train_fold = X_rf.index[train_idx_rf]
                mase_rf = np.nan
                if not original_series_indices_for_train_fold.empty:
                    end_index_for_naive_train = original_series_indices_for_train_fold[-1]
                    original_series_for_naive_train = df_city_data_input_rf[target_col_rf].iloc[:end_index_for_naive_train + 1]
                    if len(original_series_for_naive_train) > 1:
                         mase_rf = calculate_mase_pred_rf(y_test_rf.values, predictions_rf, original_series_for_naive_train.values)

                horizon_maes_rf.append(mae_rf); horizon_rmses_rf.append(rmse_rf)
                if not np.isnan(mase_rf): horizon_mases_rf.append(mase_rf)
            except Exception as e_rf_fold:
                # print(f"        Error en Fold {fold_rf+1} para RF (H={horizon_rf}h): {e_rf_fold}")
                continue
        
        results_by_horizon_rf.append({
            'Horizon': horizon_rf,
            'MAE': np.nanmean(horizon_maes_rf) if horizon_maes_rf else np.nan,
            'RMSE': np.nanmean(horizon_rmses_rf) if horizon_rmses_rf else np.nan,
            'MASE': np.nanmean(horizon_mases_rf) if horizon_mases_rf else np.nan
        })
    return results_by_horizon_rf

# --- Script Principal de Predicción RF ---
# ... (Carga de df_original_full_rf y definición de datasets_to_predict_with_rf sin cambios) ...
all_prediction_metrics_rf_detailed = []
script_start_time_rf_pred = time.time()

try:
    df_original_full_rf = pd.read_csv(ORIGINAL_DATA_FILE_RF_PRED)
    df_original_full_rf['Date'] = pd.to_datetime(df_original_full_rf['Date'])
    if 'CO2' in df_original_full_rf.columns: df_original_full_rf = df_original_full_rf.drop(columns=['CO2'])
    df_original_full_rf['Hour'] = df_original_full_rf['Date'].dt.hour
    df_original_full_rf['DayOfWeek'] = df_original_full_rf['Date'].dt.dayofweek
    df_original_full_rf['Month'] = df_original_full_rf['Date'].dt.month
    print(f"DataFrame original '{ORIGINAL_DATA_FILE_RF_PRED}' cargado y preprocesado para predicción RF.")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo original '{ORIGINAL_DATA_FILE_RF_PRED}'.")
    df_original_full_rf = None

datasets_to_predict_with_rf = [
    ("Original_Clean", ORIGINAL_DATA_DIR_RF_PRED, ".csv"),
    ("Imputed_RandomForestImputer_withLags", IMPUTED_RF_CSVS_ROOT_DIR, "_imputed_rf.csv"),
    ("Simulated_NaNs", SIMULATED_NANS_CSVS_ROOT_DIR, "_with_nans.csv"),
]
# ... (resto del script principal con los bucles, adaptando la llamada a train_predict_eval_rf_multi_horizon)

for dataset_type_rf, base_path_rf, file_suffix_rf in datasets_to_predict_with_rf:
    print(f"\n--- Predicción RF con Datos: {dataset_type_rf} ---")
    if not os.path.exists(base_path_rf) and dataset_type_rf != "Original_Clean":
        print(f"Directorio base no encontrado para {dataset_type_rf}: {base_path_rf}. Omitiendo.")
        continue

    if dataset_type_rf == "Original_Clean":
        if df_original_full_rf is None: continue
        if TARGET_CITY_RF_PRED not in df_original_full_rf['City'].unique():
            print(f"  Ciudad {TARGET_CITY_RF_PRED} no encontrada. Omitiendo Original_Clean.")
            continue
        city_name_rf = TARGET_CITY_RF_PRED
        print(f"  Ciudad: {city_name_rf} (Dataset: {dataset_type_rf})")
        
        # df_city_data_for_pred_rf contendrá TODAS las columnas necesarias (AQI, Contaminantes, Tiempo)
        df_city_data_for_pred_rf = df_original_full_rf[df_original_full_rf['City'] == city_name_rf][COLUMNS_TO_LOAD_RF_PRED].copy().reset_index(drop=True)
        
        # Rellenar NaNs en TODAS las columnas que se usarán (AQI y exógenas)
        for col_to_check in [TARGET_VARIABLE_RF_PRED] + CONTAMINANT_COLS_FOR_FEATURES:
            if col_to_check in df_city_data_for_pred_rf.columns and df_city_data_for_pred_rf[col_to_check].isnull().any():
                df_city_data_for_pred_rf[col_to_check] = df_city_data_for_pred_rf[col_to_check].ffill().bfill()
            if col_to_check in df_city_data_for_pred_rf.columns and df_city_data_for_pred_rf[col_to_check].isnull().any(): # Si aún hay NaNs
                print(f"    No se pudieron rellenar NaNs en {col_to_check} para {city_name_rf} ({dataset_type_rf}). Omitiendo.")
                # Marcar para omitir esta ejecución de Original_Clean
                df_city_data_for_pred_rf = pd.DataFrame() # Vaciar para que la siguiente comprobación lo salte
                break 
        if df_city_data_for_pred_rf.empty: continue


        horizon_metrics_rf = train_predict_eval_rf_multi_horizon(
            df_city_data_for_pred_rf, TARGET_VARIABLE_RF_PRED, CONTAMINANT_COLS_FOR_FEATURES, TIME_FEATURES, 
            N_LAGS_AQI, N_LAGS_EXO,
            PREDICTION_HORIZONS_RF_PRED, N_CV_SPLITS_RF_PRED
        )
        for hm_rf in horizon_metrics_rf: # ... (resto del almacenamiento de métricas)
            all_prediction_metrics_rf_detailed.append({'Dataset_Type': dataset_type_rf, 'City': city_name_rf, 'Scenario': 'N/A_Original','Repetition_File_ID': 'N/A_Original', 'Horizon_h': hm_rf['Horizon'],'MAE': hm_rf['MAE'], 'RMSE': hm_rf['RMSE'], 'MASE': hm_rf['MASE']})
        if horizon_metrics_rf and horizon_metrics_rf[0].get('MAE') is not np.nan:
            print(f"    Métricas RF ({dataset_type_rf}, H={horizon_metrics_rf[0]['Horizon']}h): MAE={horizon_metrics_rf[0]['MAE']:.4f}, RMSE={horizon_metrics_rf[0]['RMSE']:.4f}, MASE={horizon_metrics_rf[0]['MASE']:.4f}")
            if len(horizon_metrics_rf) > 1 and horizon_metrics_rf[-1].get('MAE') is not np.nan:
                print(f"    Métricas RF ({dataset_type_rf}, H={horizon_metrics_rf[-1]['Horizon']}h): MAE={horizon_metrics_rf[-1]['MAE']:.4f}, RMSE={horizon_metrics_rf[-1]['RMSE']:.4f}, MASE={horizon_metrics_rf[-1]['MASE']:.4f}")
        continue

    city_folder_name_safe_rf_target = TARGET_CITY_RF_PRED.replace(" ", "_").replace("-", "_")
    city_path_rf = os.path.join(base_path_rf, city_folder_name_safe_rf_target)
    if not os.path.isdir(city_path_rf):
        print(f"  Carpeta de la ciudad {TARGET_CITY_RF_PRED} no encontrada en {base_path_rf} para {dataset_type_rf}. Omitiendo.")
        continue
    original_city_name_rf = TARGET_CITY_RF_PRED
    print(f"  Ciudad: {original_city_name_rf} (Dataset: {dataset_type_rf}, desde carpeta: {city_folder_name_safe_rf_target})")

    for scenario_folder_name_safe_rf in os.listdir(city_path_rf):
        scenario_path_rf = os.path.join(city_path_rf, scenario_folder_name_safe_rf)
        if not os.path.isdir(scenario_path_rf): continue
        print(f"    Escenario: {scenario_folder_name_safe_rf}")
        pattern_rf = os.path.join(scenario_path_rf, f"*{file_suffix_rf}")
        repetition_files_rf = sorted(glob.glob(pattern_rf))
        if not repetition_files_rf: continue
        scenario_horizon_metrics_accumulator_rf = {h: {'maes': [], 'rmses': [], 'mases': []} for h in PREDICTION_HORIZONS_RF_PRED}

        for rep_file_path_rf in repetition_files_rf: # ... (resto del bucle de repeticiones)
            file_name_rf = os.path.basename(rep_file_path_rf)
            try:
                rep_file_id_rf = "_".join(file_name_rf.split('_')[-4:-2]) if "_rep" in file_name_rf and "_seed" in file_name_rf else file_name_rf
                df_rep_rf_raw = pd.read_csv(rep_file_path_rf, usecols=lambda c: c in COLUMNS_TO_LOAD_RF_PRED)
            except Exception as e_load_rf: print(f"        Error cargando {rep_file_path_rf}: {e_load_rf}. Omitiendo."); continue
            
            if not all(tf in df_rep_rf_raw.columns for tf in TIME_FEATURES):
                if 'Date' not in df_rep_rf_raw.columns: print(f"        Advertencia: 'Date' no en {file_name_rf}. Omitiendo."); continue
                temp_dates_rf = pd.to_datetime(df_rep_rf_raw['Date'])
                for tf_val_rf in TIME_FEATURES:
                    if tf_val_rf == 'Hour': df_rep_rf_raw['Hour'] = temp_dates_rf.dt.hour
                    elif tf_val_rf == 'DayOfWeek': df_rep_rf_raw['DayOfWeek'] = temp_dates_rf.dt.dayofweek
                    elif tf_val_rf == 'Month': df_rep_rf_raw['Month'] = temp_dates_rf.dt.month
            
            df_city_rep_pred_series_rf = df_rep_rf_raw[df_rep_rf_raw['City'] == original_city_name_rf][COLUMNS_TO_LOAD_RF_PRED].copy().reset_index(drop=True) # Cargar todas las columnas necesarias
            
            # Para Simulated_NaNs o cualquier dataset que pueda tener NaNs, rellenar ANTES de pasar a la función
            for col_to_check_pred in [TARGET_VARIABLE_RF_PRED] + CONTAMINANT_COLS_FOR_FEATURES:
                 if col_to_check_pred in df_city_rep_pred_series_rf.columns and df_city_rep_pred_series_rf[col_to_check_pred].isnull().any():
                    df_city_rep_pred_series_rf[col_to_check_pred] = df_city_rep_pred_series_rf[col_to_check_pred].ffill().bfill()
            # Verificar si el relleno fue exitoso para la variable objetivo
            if df_city_rep_pred_series_rf[TARGET_VARIABLE_RF_PRED].isnull().any(): continue
            
            horizon_metrics_rep_rf = train_predict_eval_rf_multi_horizon(
                df_city_rep_pred_series_rf, TARGET_VARIABLE_RF_PRED, CONTAMINANT_COLS_FOR_FEATURES, TIME_FEATURES,
                N_LAGS_AQI, N_LAGS_EXO,
                PREDICTION_HORIZONS_RF_PRED, N_CV_SPLITS_RF_PRED
            )
            for hm_rep_rf in horizon_metrics_rep_rf: # ... (resto del almacenamiento y acumulación de métricas)
                h_rf_val = hm_rep_rf['Horizon']
                all_prediction_metrics_rf_detailed.append({'Dataset_Type': dataset_type_rf, 'City': original_city_name_rf, 'Scenario': scenario_folder_name_safe_rf,'Repetition_File_ID': rep_file_id_rf, 'Horizon_h': h_rf_val,'MAE': hm_rep_rf['MAE'], 'RMSE': hm_rep_rf['RMSE'], 'MASE': hm_rep_rf['MASE']})
                if not np.isnan(hm_rep_rf.get('MAE')): scenario_horizon_metrics_accumulator_rf[h_rf_val]['maes'].append(hm_rep_rf['MAE'])
                if not np.isnan(hm_rep_rf.get('RMSE')): scenario_horizon_metrics_accumulator_rf[h_rf_val]['rmses'].append(hm_rep_rf['RMSE'])
                if not np.isnan(hm_rep_rf.get('MASE')): scenario_horizon_metrics_accumulator_rf[h_rf_val]['mases'].append(hm_rep_rf['MASE'])
        
        if any(scenario_horizon_metrics_accumulator_rf[h]['maes'] for h in scenario_horizon_metrics_accumulator_rf): # ... (resto del print de promedios)
            print(f"      Promedios RF para Escenario '{scenario_folder_name_safe_rf}':")
            for h_summary_rf in [PREDICTION_HORIZONS_RF_PRED[0], PREDICTION_HORIZONS_RF_PRED[-1]]:
                 if h_summary_rf in scenario_horizon_metrics_accumulator_rf and scenario_horizon_metrics_accumulator_rf[h_summary_rf]['maes']:
                    avg_rf_mae = np.nanmean(scenario_horizon_metrics_accumulator_rf[h_summary_rf]['maes']); avg_rf_rmse = np.nanmean(scenario_horizon_metrics_accumulator_rf[h_summary_rf]['rmses']); avg_rf_mase = np.nanmean(scenario_horizon_metrics_accumulator_rf[h_summary_rf]['mases'])
                    if not (np.isnan(avg_rf_mae) and np.isnan(avg_rf_rmse) and np.isnan(avg_rf_mase)):
                        print(f"        Horizonte {h_summary_rf}h: MAE={avg_rf_mae:.4f}, RMSE={avg_rf_rmse:.4f}, MASE={avg_rf_mase:.4f}")

# --- Guardado de Métricas de Predicción RF (sin cambios en esta sección) ---
# ... (copiar la sección de guardado de métricas del script anterior de predicción RF) ...
df_all_prediction_metrics_rf_detailed = pd.DataFrame(all_prediction_metrics_rf_detailed)
if not df_all_prediction_metrics_rf_detailed.empty:
    detailed_output_path_rf = os.path.join(PREDICTION_METRICS_RF_ROOT_DIR, f'{TARGET_CITY_RF_PRED}_prediction_metrics_detailed_{PREDICTION_METHOD_NAME_RF_PRED}.csv')
    df_all_prediction_metrics_rf_detailed.to_csv(detailed_output_path_rf, index=False)
    print(f"\nMétricas de predicción RF detalladas (para {TARGET_CITY_RF_PRED}) guardadas en: {detailed_output_path_rf}")
    if not df_all_prediction_metrics_rf_detailed.empty: print(df_all_prediction_metrics_rf_detailed.head())
    city_name_rf_file_summary = TARGET_CITY_RF_PRED.replace(" ", "_").replace("-", "_")
    city_metrics_dir_rf_summary = os.path.join(PREDICTION_METRICS_RF_ROOT_DIR, city_name_rf_file_summary)
    os.makedirs(city_metrics_dir_rf_summary, exist_ok=True)
    df_summary_for_target_city_rf = df_all_prediction_metrics_rf_detailed[df_all_prediction_metrics_rf_detailed['City'] == TARGET_CITY_RF_PRED]
    if not df_summary_for_target_city_rf.empty:
        df_summary_by_dataset_scenario_horizon_rf = df_summary_for_target_city_rf.groupby(
            ['Dataset_Type', 'Scenario', 'Horizon_h']
        ).agg(
            Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean'),
            Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Std_MASE=('MASE', 'std'),
            Count_Repetitions=('MAE', 'size')
        ).reset_index()
        city_summary_file_path_rf = os.path.join(city_metrics_dir_rf_summary, f'metrics_summary_{PREDICTION_METHOD_NAME_RF_PRED}_{city_name_rf_file_summary}.csv')
        df_summary_by_dataset_scenario_horizon_rf.to_csv(city_summary_file_path_rf, index=False)
        print(f"  Resumen de métricas RF para {TARGET_CITY_RF_PRED} guardado en: {city_summary_file_path_rf}")
    df_global_summary_by_dataset_horizon_rf = df_all_prediction_metrics_rf_detailed.groupby(
        ['Dataset_Type', 'Horizon_h']
    ).agg(
        Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')
    ).reset_index()
    global_summary_file_path_rf = os.path.join(PREDICTION_METRICS_RF_ROOT_DIR, f'{TARGET_CITY_RF_PRED}_prediction_metrics_global_summary_{PREDICTION_METHOD_NAME_RF_PRED}.csv')
    df_global_summary_by_dataset_horizon_rf.to_csv(global_summary_file_path_rf, index=False)
    print(f"\nResumen 'global' (para {TARGET_CITY_RF_PRED}) de métricas RF promedio guardado en: {global_summary_file_path_rf}")
    if not df_global_summary_by_dataset_horizon_rf.empty: print(df_global_summary_by_dataset_horizon_rf)

script_end_time_rf_pred = time.time()
print(f"\nScript de predicción RF completado. Tiempo total: {(script_end_time_rf_pred - script_start_time_rf_pred)/60:.2f} minutos.")

DataFrame original 'c:\Users\fdemiguel\Documentos\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' cargado y preprocesado para predicción RF.

--- Predicción RF con Datos: Original_Clean ---
  Ciudad: Brasilia (Dataset: Original_Clean)
    Métricas RF (Original_Clean, H=1h): MAE=2.1485, RMSE=3.4335, MASE=1.3119
    Métricas RF (Original_Clean, H=24h): MAE=4.1897, RMSE=6.0334, MASE=2.5496

--- Predicción RF con Datos: Imputed_RandomForestImputer_withLags ---
  Ciudad: Brasilia (Dataset: Imputed_RandomForestImputer_withLags, desde carpeta: Brasilia)
    Escenario: BlockRows_12h
      Promedios RF para Escenario 'BlockRows_12h':
        Horizonte 1h: MAE=2.1719, RMSE=3.4766, MASE=1.3286
        Horizonte 24h: MAE=4.2162, RMSE=6.0318, MASE=2.5718
    Escenario: BlockRows_24h
      Promedios RF para Escenario 'BlockRows_24h':
        Horizonte 1h: MAE=2.1622, RMSE=3.4781, MASE=1.3210
        Horizonte 24h: MAE=4.1882, RMSE=6.0402, MASE=2.5497
    Escenario: BlockRows_2h
      Promedios RF pa

LSTM-VAE

Cambiar csv de rutas y tal, CAMBIAR DEL SCRIPT LSTM-VAE DE IMPUTACION

In [3]:
import pandas as pd
import numpy as np
import os
import glob
import re
from unicodedata import normalize

# --- Helper Function for Sanitizing TF Names ---
def sanitize_tf_name(name_str):
    s = normalize('NFKD', name_str); s = s.encode('ascii', 'ignore'); s = s.decode('ascii')
    s = re.sub(r'[^a-zA-Z0-9_]', '_', s);
    if s and s[0].isdigit() and not s.isdigit(): s = "n_" + s
    if not s: s = "sanitized_default_name"
    return s

# --- 0. Carga de Datos Originales COMPLETOS (para TIME_FEATURES y referencia) ---
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv') # Asume que está en la subcarpeta CSVs
    if not os.path.exists(file_path_load_orig):
        # Intento alternativo si el script está en una subcarpeta y CSVs está un nivel arriba
        script_dir_parent_orig = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent_orig, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig):
            file_path_load_orig = alt_file_path_load_orig
        else:
            raise FileNotFoundError("Air_Quality.csv (original completo) no encontrado.")
    df_air_quality_full_original_unify = pd.read_csv(file_path_load_orig) # Nuevo nombre de variable
    print(f"Datos originales completos cargados desde: {file_path_load_orig} para unificación.")
except FileNotFoundError:
    print(f"Error: Air_Quality.csv (original completo) no encontrado.")
    print("Creando DataFrame de ejemplo para unificación...")
    _example_data_orig = []
    _cities_example_orig = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New_York', 'Sydney']
    _start_date_orig = pd.to_datetime('2024-01-01')
    # Crear suficientes datos para cubrir la longitud esperada
    num_example_rows = 8784 # Un año de datos horarios como en tu dataset original
    for _city_ex_orig in _cities_example_orig:
        for i in range(num_example_rows):
            _date_orig = _start_date_orig + pd.Timedelta(hours=i)
            _example_data_orig.append([
                _date_orig, _city_ex_orig,
                200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5,
                50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20,
                40+np.random.rand()*20
            ])
    df_air_quality_full_original_unify = pd.DataFrame(_example_data_orig, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo (original completo) creado para unificación.")

df_air_quality_full_original_unify['Date'] = pd.to_datetime(df_air_quality_full_original_unify['Date'])
if 'CO2' in df_air_quality_full_original_unify.columns:
    df_air_quality_full_original_unify = df_air_quality_full_original_unify.drop(columns=['CO2'])
# Crear TIME_FEATURES en el DataFrame original completo
df_air_quality_full_original_unify['Hour'] = df_air_quality_full_original_unify['Date'].dt.hour
df_air_quality_full_original_unify['DayOfWeek'] = df_air_quality_full_original_unify['Date'].dt.dayofweek
df_air_quality_full_original_unify['Month'] = df_air_quality_full_original_unify['Date'].dt.month
TIME_FEATURES_UNIFY = ['Hour', 'DayOfWeek', 'Month']


# --- Configuración ---
TARGET_CITY_UNIFY = "Brasilia"
ALL_TARGET_COLS_FOR_UNIFY = ['AQI', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']

NAN_SIMULATION_SCENARIOS_UNIFY = [
    {'name': 'RandomRows_10percent'}, {'name': 'RandomRows_20percent'},
    {'name': 'BlockRows_2h'}, {'name': 'BlockRows_4h'}, {'name': 'BlockRows_8h'},
    {'name': 'BlockRows_12h'}, {'name': 'BlockRows_24h'},
]
N_REPETITIONS_UNIFY = 1

PROJECT_ROOT_DIR_UNIFY = os.getcwd()
LSTM_VAE_IMPUTED_INDIVIDUAL_VARS_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_UNIFY, 'imputaciones', 'LSTM_VAE_imputer', 'CSVs')
LSTM_VAE_IMPUTED_UNIFIED_CSVS_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_UNIFY, 'imputaciones', 'LSTM_VAE_imputer', 'CSVs_unified')
os.makedirs(LSTM_VAE_IMPUTED_UNIFIED_CSVS_ROOT_DIR, exist_ok=True)
SIMULATED_DATA_WITH_NANS_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_UNIFY, 'CSVs', 'simulados')

print(f"--- Iniciando Unificación de CSVs Imputados por LSTM-VAE para {TARGET_CITY_UNIFY} ---")
city_name_safe_unify = TARGET_CITY_UNIFY.replace(" ", "_").replace("-", "_")
city_individual_vars_path = os.path.join(LSTM_VAE_IMPUTED_INDIVIDUAL_VARS_ROOT_DIR, city_name_safe_unify)

if not os.path.exists(city_individual_vars_path):
    print(f"Error: Directorio de entrada para {TARGET_CITY_UNIFY} no encontrado: {city_individual_vars_path}")
    exit()

city_unified_output_path = os.path.join(LSTM_VAE_IMPUTED_UNIFIED_CSVS_ROOT_DIR, city_name_safe_unify)
os.makedirs(city_unified_output_path, exist_ok=True)

# Filtrar el DataFrame original completo una vez para la ciudad objetivo
df_original_city_full_sample_unify = df_air_quality_full_original_unify[
    df_air_quality_full_original_unify['City'] == TARGET_CITY_UNIFY
].copy().reset_index(drop=True) # Asegurar que el índice sea contiguo desde 0

if df_original_city_full_sample_unify.empty:
    print(f"No hay datos en el DataFrame original completo para la ciudad {TARGET_CITY_UNIFY}. No se pueden añadir TIME_FEATURES consistentemente.")
    # Podrías decidir salir o continuar sin añadir TIME_FEATURES si no se cargan.
    # Por ahora, continuaremos, pero es una advertencia importante.


for scenario_info in NAN_SIMULATION_SCENARIOS_UNIFY:
    scenario_name_orig = scenario_info['name']
    scenario_name_safe_unify = scenario_name_orig.replace("%", "percent").replace(" ", "_")
    print(f"  Procesando Escenario: {scenario_name_orig}")
    scenario_unified_output_path = os.path.join(city_unified_output_path, scenario_name_safe_unify)
    os.makedirs(scenario_unified_output_path, exist_ok=True)

    for rep in range(1, N_REPETITIONS_UNIFY + 1):
        print(f"    Repetición: {rep}")
        df_unified_rep = None
        first_file_processed = False
        seed_loaded_for_rep = None
        base_df_length = 0

        original_sim_data_path = os.path.join(SIMULATED_DATA_WITH_NANS_ROOT_DIR, city_name_safe_unify, scenario_name_safe_unify)
        original_sim_files = []
        if os.path.exists(original_sim_data_path):
            original_sim_files = sorted([
                f for f in os.listdir(original_sim_data_path)
                if f.startswith(f"{city_name_safe_unify}_{scenario_name_safe_unify}_rep{rep}") and f.endswith("_with_nans.csv")
            ])
        
        if original_sim_files:
            base_sim_file_path = os.path.join(original_sim_data_path, original_sim_files[0])
            try:
                # Cargar solo Date y City, o todas si es necesario para la longitud
                df_base_structure_loaded = pd.read_csv(base_sim_file_path)
                df_unified_rep = df_base_structure_loaded[['Date', 'City']].copy()
                base_df_length = len(df_unified_rep)
                seed_loaded_for_rep = original_sim_files[0].split('_seed')[-1].split('_with_nans.csv')[0]
                first_file_processed = True
            except Exception as e:
                print(f"      Error cargando archivo base simulado {original_sim_files[0]}: {e}.")
                df_unified_rep = pd.DataFrame()
        else:
            print(f"      Advertencia: No se encontró _with_nans.csv para {scenario_name_safe_unify} rep {rep}.")
            df_unified_rep = pd.DataFrame()

        for target_var in ALL_TARGET_COLS_FOR_UNIFY:
            target_var_safe = sanitize_tf_name(target_var)
            var_scenario_path = os.path.join(city_individual_vars_path, target_var_safe, scenario_name_safe_unify)
            if not os.path.exists(var_scenario_path):
                print(f"      Directorio no encontrado para var {target_var}, esc {scenario_name_safe_unify}. Omitiendo var.")
                if target_var not in df_unified_rep.columns and first_file_processed: df_unified_rep[target_var] = np.nan
                continue
            file_pattern_var_rep = f"{city_name_safe_unify}_{scenario_name_safe_unify}_rep{rep}_seed*imputed_vae_{target_var_safe}.csv"
            matching_var_files = sorted(glob.glob(os.path.join(var_scenario_path, file_pattern_var_rep)))
            if not matching_var_files:
                print(f"      No se encontró archivo para var {target_var}, esc {scenario_name_safe_unify}, rep {rep}. Omitiendo var.")
                if target_var not in df_unified_rep.columns and first_file_processed: df_unified_rep[target_var] = np.nan
                continue
            var_file_path = matching_var_files[0]
            try:
                df_var_imputed = pd.read_csv(var_file_path)
                if not first_file_processed: # Si el _with_nans no se cargó
                    if 'Date' in df_var_imputed.columns and 'City' in df_var_imputed.columns:
                        df_unified_rep['Date'] = df_var_imputed['Date']
                        df_unified_rep['City'] = df_var_imputed['City']
                        base_df_length = len(df_unified_rep)
                        seed_loaded_for_rep = var_file_path.split('_seed')[-1].split(f'_imputed_vae_{target_var_safe}.csv')[0]
                        first_file_processed = True
                    else:
                        df_unified_rep = None; break
                if target_var in df_var_imputed.columns:
                    if not df_unified_rep.empty and len(df_unified_rep) != len(df_var_imputed) and base_df_length > 0:
                        print(f"      Advertencia: Discrepancia longitud para {target_var}. Alineando a {base_df_length}.")
                        df_unified_rep[target_var] = df_var_imputed[target_var].iloc[:base_df_length].values
                    elif base_df_length > 0: # Si df_unified_rep estaba vacío pero ahora tenemos base_df_length
                         df_unified_rep[target_var] = df_var_imputed[target_var].iloc[:base_df_length].values
                    else: # Si no hay base_df_length (error cargando with_nans y primer archivo de variable)
                         df_unified_rep[target_var] = df_var_imputed[target_var]

                else:
                    if target_var not in df_unified_rep.columns and first_file_processed: df_unified_rep[target_var] = np.nan
            except Exception as e:
                print(f"      Error procesando {var_file_path}: {e}")
                if target_var not in df_unified_rep.columns and df_unified_rep is not None and not df_unified_rep.empty:
                    df_unified_rep[target_var] = np.nan
        
        if df_unified_rep is not None and not df_unified_rep.empty and seed_loaded_for_rep is not None:
            final_cols_ordered = ['Date', 'City'] + [col for col in ALL_TARGET_COLS_FOR_UNIFY if col in df_unified_rep.columns]
            
            # Añadir TIME_FEATURES usando el df_original_city_full_sample_unify
            # que ya está filtrado por ciudad y tiene índice reseteado
            if not df_original_city_full_sample_unify.empty:
                # Asegurar que la longitud de df_unified_rep no exceda la del original sample
                current_len_unified = len(df_unified_rep)
                for tf_col in TIME_FEATURES_UNIFY:
                    if tf_col not in df_unified_rep.columns: # Solo añadir si no existe
                        if len(df_original_city_full_sample_unify) >= current_len_unified:
                            df_unified_rep[tf_col] = df_original_city_full_sample_unify[tf_col].iloc[:current_len_unified].values
                        else:
                            print(f"    Advertencia: df_original_city_full_sample_unify más corto que df_unified_rep. No se pueden añadir todas las TIME_FEATURES para {tf_col}.")
                    if tf_col not in final_cols_ordered: # Añadir a la lista de orden final
                        final_cols_ordered.append(tf_col)
            else:
                print(f"    Advertencia: No se pudieron añadir TIME_FEATURES porque df_original_city_full_sample_unify está vacío para {TARGET_CITY_UNIFY}.")


            df_unified_rep = df_unified_rep[final_cols_ordered]
            unified_filename = f"{city_name_safe_unify}_{scenario_name_safe_unify}_rep{rep}_seed{seed_loaded_for_rep}_imputed_vae_ALL.csv"
            unified_save_path = os.path.join(scenario_unified_output_path, unified_filename)
            df_unified_rep.to_csv(unified_save_path, index=False)
            print(f"      Archivo unificado guardado: {unified_filename}")
        elif seed_loaded_for_rep is None:
             print(f"      No se pudo determinar semilla para rep {rep}, esc {scenario_name_safe_unify}. No se guardó unificado.")
        else:
            print(f"      No se generó DataFrame unificado para rep {rep}, esc {scenario_name_safe_unify}.")

print(f"\n--- Unificación Completada para {TARGET_CITY_UNIFY} ---")
print(f"Archivos unificados guardados en: {LSTM_VAE_IMPUTED_UNIFIED_CSVS_ROOT_DIR}")

Datos originales completos cargados desde: CSVs\Air_Quality.csv para unificación.
--- Iniciando Unificación de CSVs Imputados por LSTM-VAE para Brasilia ---
  Procesando Escenario: RandomRows_10percent
    Repetición: 1
      Archivo unificado guardado: Brasilia_RandomRows_10percent_rep1_seed2377262672_imputed_vae_ALL.csv
  Procesando Escenario: RandomRows_20percent
    Repetición: 1
      Archivo unificado guardado: Brasilia_RandomRows_20percent_rep1_seed4250581208_imputed_vae_ALL.csv
  Procesando Escenario: BlockRows_2h
    Repetición: 1
      Archivo unificado guardado: Brasilia_BlockRows_2h_rep1_seed2237180940_imputed_vae_ALL.csv
  Procesando Escenario: BlockRows_4h
    Repetición: 1
      Archivo unificado guardado: Brasilia_BlockRows_4h_rep1_seed3793552506_imputed_vae_ALL.csv
  Procesando Escenario: BlockRows_8h
    Repetición: 1
      Archivo unificado guardado: Brasilia_BlockRows_8h_rep1_seed2002168121_imputed_vae_ALL.csv
  Procesando Escenario: BlockRows_12h
    Repetición: 1


In [7]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import tensorflow as tf
import keras # Keras 3 de alto nivel
from keras import layers, Model, optimizers, regularizers
from keras import backend as K
import time
import glob
import warnings
import shutil
from unicodedata import normalize
import re

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Helper Function for Sanitizing TF Names ---
def sanitize_tf_name(name_str):
    s = normalize('NFKD', name_str); s = s.encode('ascii', 'ignore'); s = s.decode('ascii')
    s = re.sub(r'[^a-zA-Z0-9_]', '_', s)
    if s and s[0].isdigit() and not s.isdigit(): s = "n_" + s
    if not s: s = "sanitized_default_name"
    return s

# --- 0. Configuración Inicial y Parámetros ---
TARGET_CITY_LSTM_PRED = "Brasilia"
TARGET_VARIABLE_LSTM_PRED = 'AQI'
EXOGENOUS_COLS_LSTM_PRED = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES_LSTM_PRED = ['Hour', 'DayOfWeek', 'Month']

N_LAGS_INPUT_LSTM_PRED = 24 * 1
PREDICTION_HORIZONS_LSTM_PRED = [1, 2, 4, 8, 12, 24]
MAX_HORIZON_LSTM_PRED = max(PREDICTION_HORIZONS_LSTM_PRED)

LSTM_UNITS_L1_PRED = 32
LSTM_UNITS_L2_PRED = 16
DENSE_UNITS_PRED = 16
DROPOUT_RATE_PRED = 0.2
INITIAL_LR_PRED = 1e-3
BATCH_SIZE_PRED = 64
EPOCHS_PER_FOLD_PRED = 30 # Mantener reducido para pruebas
ES_PATIENCE_PRED = 10
LR_PATIENCE_PRED = 5

N_CV_SPLITS_LSTM_PRED = 2 # Mantener reducido para pruebas
N_REPETITIONS_LSTM_PRED_FOR_SCENARIOS = 1 # Mantener en 1 para pruebas

COLUMNS_TO_LOAD_LSTM_PRED = ['Date', 'City', TARGET_VARIABLE_LSTM_PRED] + EXOGENOUS_COLS_LSTM_PRED + TIME_FEATURES_LSTM_PRED
COLUMNS_TO_LOAD_LSTM_PRED = sorted(list(set(COLUMNS_TO_LOAD_LSTM_PRED)))

# --- Rutas ---
PROJECT_ROOT_DIR_LSTM_PRED = os.getcwd()
ORIGINAL_DATA_DIR_LSTM_PRED = os.path.join(PROJECT_ROOT_DIR_LSTM_PRED, 'CSVs')
ORIGINAL_DATA_FILE_LSTM_PRED = os.path.join(ORIGINAL_DATA_DIR_LSTM_PRED, 'Air_Quality.csv')
IMPUTATIONS_BASE_INPUT_DIR_LSTM_PRED = os.path.join(PROJECT_ROOT_DIR_LSTM_PRED, 'imputaciones')
IMPUTED_LINEAR_CSVS_ROOT_DIR_LP = os.path.join(IMPUTATIONS_BASE_INPUT_DIR_LSTM_PRED, 'interpolacionLineal', 'CSVs')
IMPUTED_SARIMA_CSVS_ROOT_DIR_SP = os.path.join(IMPUTATIONS_BASE_INPUT_DIR_LSTM_PRED, 'SARIMA', 'CSVs') # Ajustado nombre de carpeta SARIMA
IMPUTED_LSTM_VAE_CSVS_ROOT_DIR_VP = os.path.join(IMPUTATIONS_BASE_INPUT_DIR_LSTM_PRED, 'LSTM_VAE_imputer', 'CSVs') # Usar los unificados

PREDICTION_METHOD_NAME_LSTM_PRED = 'LSTM_VAE_Pred'
PREDICTION_METRICS_LSTM_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_LSTM_PRED, 'predicciones', PREDICTION_METHOD_NAME_LSTM_PRED, 'metricas')
os.makedirs(PREDICTION_METRICS_LSTM_ROOT_DIR, exist_ok=True)

# --- Funciones Auxiliares (sin cambios lógicos internos) ---
def create_sequences_for_forecasting(df_scaled_features, target_col_name_scaled, n_lags_input, list_of_horizons_to_predict):
    X_list, Y_multih_list = [], []
    data_values = df_scaled_features.values
    target_col_idx = df_scaled_features.columns.get_loc(target_col_name_scaled)
    max_h = max(list_of_horizons_to_predict)

    for i in range(len(data_values) - n_lags_input - max_h + 1):
        X_list.append(data_values[i : i + n_lags_input, :])
        # Recolectar y_target para cada horizonte en la lista
        current_y_targets = []
        for h_val in list_of_horizons_to_predict:
            # El índice para y es i (inicio de ventana X) + n_lags_input (fin de ventana X) + h_val (horizonte) - 1 (porque es indexado 0)
            target_idx_for_h = i + n_lags_input + h_val - 1
            current_y_targets.append(data_values[target_idx_for_h, target_col_idx])
        Y_multih_list.append(current_y_targets)

    if not X_list:
        return np.array([]).reshape(0, n_lags_input, df_scaled_features.shape[1]), np.array([])
    return np.array(X_list), np.array(Y_multih_list) # Y_multih_list shape: (n_samples, len(list_of_horizons_to_predict))


def calculate_mase_pred_lstm(y_true, y_pred, y_train_for_naive_unscaled):
    if len(y_train_for_naive_unscaled) <= 1: return np.nan
    train_vals = np.asarray(y_train_for_naive_unscaled)
    if len(train_vals) <= 1: return np.nan
    mae_naive_on_train = np.mean(np.abs(np.diff(train_vals)))
    if mae_naive_on_train == 0 or np.isnan(mae_naive_on_train) or mae_naive_on_train < 1e-9 : return np.nan
    true_vals, pred_vals = np.asarray(y_true), np.asarray(y_pred)
    actual_forecast_errors = np.abs(true_vals - pred_vals)
    if len(actual_forecast_errors) == 0: return np.nan # Evitar error si está vacío
    mase = np.mean(actual_forecast_errors) / mae_naive_on_train
    return mase

def build_lstm_predictor_model(n_lags_input, n_features_input, lstm_units_l1, lstm_units_l2, dense_units, dropout_rate, num_horizon_outputs, model_name_prefix="lstm_pred"):
    K.clear_session()
    input_seq = layers.Input(shape=(n_lags_input, n_features_input), name=f"{model_name_prefix}_input")
    x = layers.LSTM(lstm_units_l1, return_sequences=(lstm_units_l2 > 0), name=f"{model_name_prefix}_lstm1")(input_seq)
    x = layers.BatchNormalization()(x) # Añadido BatchNormalization
    x = layers.Dropout(dropout_rate, name=f"{model_name_prefix}_dropout1")(x)
    if lstm_units_l2 > 0:
        x = layers.LSTM(lstm_units_l2, return_sequences=False, name=f"{model_name_prefix}_lstm2")(x)
        x = layers.BatchNormalization()(x) # Añadido BatchNormalization
        x = layers.Dropout(dropout_rate, name=f"{model_name_prefix}_dropout2")(x)
    if dense_units > 0:
        x = layers.Dense(dense_units, activation='relu', name=f"{model_name_prefix}_dense1", kernel_regularizer=regularizers.l2(0.001))(x)
        x = layers.Dropout(dropout_rate, name=f"{model_name_prefix}_dropout_dense")(x)
    output_pred = layers.Dense(num_horizon_outputs, activation='linear', name=f"{model_name_prefix}_output")(x)
    model = Model(inputs=input_seq, outputs=output_pred, name=model_name_prefix)
    return model

def train_predict_eval_lstm_multi_horizon(
    df_city_data_input_lstm, target_col_lstm, exog_cols_lstm, time_features_list_lstm,
    n_lags_input_lstm, horizons_list_lstm, n_cv_splits_lstm, model_params_lstm
):
    results_by_horizon_lstm = {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in horizons_list_lstm}
    features_for_scaling_lstm = [target_col_lstm] + exog_cols_lstm + time_features_list_lstm
    df_features_to_scale_lstm = df_city_data_input_lstm[features_for_scaling_lstm].copy()
    
    scaler_lstm = MinMaxScaler(feature_range=(0, 1))
    scaled_features_lstm_array = scaler_lstm.fit_transform(df_features_to_scale_lstm)
    df_scaled_features_lstm = pd.DataFrame(scaled_features_lstm_array, columns=features_for_scaling_lstm, index=df_features_to_scale_lstm.index)
    
    target_col_scaled_name_lstm = target_col_lstm
    min_total_len_lstm = n_lags_input_lstm + max(horizons_list_lstm) + n_cv_splits_lstm * 30 # Aumentado un poco
    if len(df_scaled_features_lstm) < min_total_len_lstm:
        print(f"      Datos insuficientes ({len(df_scaled_features_lstm)}) para LSTM con CV. Mínimo: {min_total_len_lstm}")
        return [results_by_horizon_lstm[h] for h in horizons_list_lstm]

    X_pred_np, Y_multi_horizon_np = create_sequences_for_forecasting(
        df_scaled_features_lstm, target_col_scaled_name_lstm, n_lags_input_lstm, horizons_list_lstm
    )
    if X_pred_np.shape[0] == 0 or Y_multi_horizon_np.shape[0] == 0:
        print("      No se pudieron crear secuencias para el predictor LSTM.")
        return [results_by_horizon_lstm[h] for h in horizons_list_lstm]

    tscv_lstm = TimeSeriesSplit(n_splits=n_cv_splits_lstm)
    accumulated_metrics_per_horizon_idx = {idx: {'maes': [], 'rmses': [], 'mases': []} for idx in range(len(horizons_list_lstm))}
    n_features_lstm_model_input = X_pred_np.shape[2]

    for fold_lstm, (train_idx_lstm, test_idx_lstm) in enumerate(tscv_lstm.split(X_pred_np)):
        X_train_lstm, X_test_lstm = X_pred_np[train_idx_lstm], X_pred_np[test_idx_lstm]
        Y_train_multi_lstm, Y_test_multi_lstm = Y_multi_horizon_np[train_idx_lstm], Y_multi_horizon_np[test_idx_lstm]

        if X_train_lstm.shape[0] < model_params_lstm['batch_size'] or X_test_lstm.shape[0] == 0: continue

        model_lstm_pred = build_lstm_predictor_model(
            n_lags_input_lstm, n_features_lstm_model_input,
            model_params_lstm['lstm_units_l1'], model_params_lstm['lstm_units_l2'],
            model_params_lstm['dense_units'], model_params_lstm['dropout_rate'],
            num_horizon_outputs=len(horizons_list_lstm),
            model_name_prefix=f"lstm_pred_fold{fold_lstm}"
        )
        model_lstm_pred.compile(optimizer=optimizers.AdamW(learning_rate=model_params_lstm['initial_lr'], weight_decay=0.001, clipnorm=1.0), loss='mse')
        
        # Para MASE, necesitamos la serie original (desescalada) del fold de entrenamiento
        # Encontramos el índice final de la última secuencia de entrada del conjunto de entrenamiento
        if not train_idx_lstm.size: continue # Skip fold if train_idx_lstm is empty
        last_sequence_in_train_idx_original_X = train_idx_lstm[-1]
        # La última observación usada para la ÚLTIMA feature de X_train_lstm[-1] es en df_city_data_input_lstm
        # en el índice: last_sequence_in_train_idx_original_X + n_lags_input_lstm - 1
        end_of_train_data_idx_original = last_sequence_in_train_idx_original_X + n_lags_input_lstm -1
        
        # Asegurarse de que el índice no exceda la longitud del dataframe original
        end_of_train_data_idx_original = min(end_of_train_data_idx_original, len(df_city_data_input_lstm) - 1)
        y_train_unscaled_for_mase = df_city_data_input_lstm[target_col_lstm].iloc[:end_of_train_data_idx_original +1].values


        try:
            val_data_lstm = None
            if len(X_test_lstm) >= model_params_lstm['batch_size']: # Solo usar val_data si es suficientemente grande
                val_data_lstm = (X_test_lstm, Y_test_multi_lstm)

            model_lstm_pred.fit(X_train_lstm, Y_train_multi_lstm,
                                epochs=model_params_lstm['epochs'], batch_size=model_params_lstm['batch_size'],
                                validation_data=val_data_lstm,
                                callbacks=[
                                    keras.callbacks.EarlyStopping(monitor='val_loss' if val_data_lstm else 'loss', mode='min', patience=model_params_lstm['es_patience'], restore_best_weights=True, verbose=0),
                                    keras.callbacks.ReduceLROnPlateau(monitor='val_loss' if val_data_lstm else 'loss', mode='min', factor=0.5, patience=model_params_lstm['lr_patience'], min_lr=1e-7, verbose=0)
                                ], verbose=0)
            
            predictions_scaled_multi_horizon = model_lstm_pred.predict(X_test_lstm, verbose=0)
            
            for h_idx, horizon_val in enumerate(horizons_list_lstm):
                y_true_h_scaled = Y_test_multi_lstm[:, h_idx]
                y_pred_h_scaled = predictions_scaled_multi_horizon[:, h_idx]

                dummy_array_true = np.zeros((len(y_true_h_scaled), n_features_lstm_model_input))
                dummy_array_pred = np.zeros((len(y_pred_h_scaled), n_features_lstm_model_input))
                target_original_idx_in_scaled_df = df_scaled_features_lstm.columns.get_loc(target_col_scaled_name_lstm)
                dummy_array_true[:, target_original_idx_in_scaled_df] = y_true_h_scaled
                dummy_array_pred[:, target_original_idx_in_scaled_df] = y_pred_h_scaled
                
                y_true_h_unscaled_full = scaler_lstm.inverse_transform(dummy_array_true)
                y_pred_h_unscaled_full = scaler_lstm.inverse_transform(dummy_array_pred)
                y_true_h_unscaled = y_true_h_unscaled_full[:, target_original_idx_in_scaled_df]
                y_pred_h_unscaled = y_pred_h_unscaled_full[:, target_original_idx_in_scaled_df]

                if len(y_true_h_unscaled) > 0:
                    mae_h = mean_absolute_error(y_true_h_unscaled, y_pred_h_unscaled)
                    rmse_h = np.sqrt(mean_squared_error(y_true_h_unscaled, y_pred_h_unscaled))
                    mase_h = calculate_mase_pred_lstm(y_true_h_unscaled, y_pred_h_unscaled, y_train_unscaled_for_mase)
                    
                    accumulated_metrics_per_horizon_idx[h_idx]['maes'].append(mae_h)
                    accumulated_metrics_per_horizon_idx[h_idx]['rmses'].append(rmse_h)
                    if not np.isnan(mase_h): accumulated_metrics_per_horizon_idx[h_idx]['mases'].append(mase_h)
        except Exception as e_lstm_fold:
            print(f"        Error en Fold {fold_lstm+1} LSTM: {e_lstm_fold}")
            continue
            
    final_results_list = []
    for h_idx, horizon_val in enumerate(horizons_list_lstm):
        final_results_list.append({
            'Horizon': horizon_val,
            'MAE': np.nanmean(accumulated_metrics_per_horizon_idx[h_idx]['maes']) if accumulated_metrics_per_horizon_idx[h_idx]['maes'] else np.nan,
            'RMSE': np.nanmean(accumulated_metrics_per_horizon_idx[h_idx]['rmses']) if accumulated_metrics_per_horizon_idx[h_idx]['rmses'] else np.nan,
            'MASE': np.nanmean(accumulated_metrics_per_horizon_idx[h_idx]['mases']) if accumulated_metrics_per_horizon_idx[h_idx]['mases'] else np.nan,
        })
    return final_results_list

# --- Script Principal de Predicción con LSTM ---
all_prediction_metrics_lstm_detailed = []
script_start_time_lstm_pred = time.time()

try:
    df_original_full_lstm = pd.read_csv(ORIGINAL_DATA_FILE_LSTM_PRED)
    df_original_full_lstm['Date'] = pd.to_datetime(df_original_full_lstm['Date'])
    if 'CO2' in df_original_full_lstm.columns: df_original_full_lstm = df_original_full_lstm.drop(columns=['CO2'])
    df_original_full_lstm['Hour'] = df_original_full_lstm['Date'].dt.hour
    df_original_full_lstm['DayOfWeek'] = df_original_full_lstm['Date'].dt.dayofweek
    df_original_full_lstm['Month'] = df_original_full_lstm['Date'].dt.month
    print(f"DataFrame original '{ORIGINAL_DATA_FILE_LSTM_PRED}' cargado para predicción LSTM.")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo original '{ORIGINAL_DATA_FILE_LSTM_PRED}'. Saliendo.")
    exit() # Salir si el original no se puede cargar


datasets_to_predict_with_lstm = [
    ("Original_Clean", ORIGINAL_DATA_DIR_LSTM_PRED, ".csv"),
    ("Imputed_LinearInterpolation", IMPUTED_LINEAR_CSVS_ROOT_DIR_LP, "_imputed_linear.csv"),
    ("SARIMA", IMPUTED_SARIMA_CSVS_ROOT_DIR_SP, "_imputed_sarima.csv"),
    ("LSTM_VAE_imputer", IMPUTED_LSTM_VAE_CSVS_ROOT_DIR_VP, "_imputed_vae_ALL.csv"), # Usar los unificados
]

model_params_to_use = {
    'lstm_units_l1': LSTM_UNITS_L1_PRED, 'lstm_units_l2': LSTM_UNITS_L2_PRED,
    'dense_units': DENSE_UNITS_PRED, 'dropout_rate': DROPOUT_RATE_PRED,
    'initial_lr': INITIAL_LR_PRED, 'batch_size': BATCH_SIZE_PRED,
    'epochs': EPOCHS_PER_FOLD_PRED, 'es_patience': ES_PATIENCE_PRED, 'lr_patience': LR_PATIENCE_PRED
}

city_name_lstm_target = TARGET_CITY_LSTM_PRED
city_name_safe_lstm_target = city_name_lstm_target.replace(" ", "_").replace("-", "_")

for dataset_type_lstm, base_path_lstm, file_suffix_lstm in datasets_to_predict_with_lstm:
    print(f"\n--- Predicción LSTM con Datos: {dataset_type_lstm} para {city_name_lstm_target} ---")

    df_city_data_for_pred_lstm = None # Inicializar
    scenario_name_lstm = 'N/A'
    rep_id_lstm = 'N/A'

    if dataset_type_lstm == "Original_Clean":
        if city_name_lstm_target not in df_original_full_lstm['City'].unique():
            print(f"  Ciudad {city_name_lstm_target} no encontrada en datos originales. Omitiendo."); continue
        df_city_data_for_pred_lstm = df_original_full_lstm[df_original_full_lstm['City'] == city_name_lstm_target][COLUMNS_TO_LOAD_LSTM_PRED].copy().reset_index(drop=True)
        scenario_name_lstm, rep_id_lstm = 'N/A_Original', 'N/A_Original'
    
    else: # Para datasets imputados
        city_path_lstm_data = os.path.join(base_path_lstm, city_name_safe_lstm_target) # Ruta a la carpeta de la ciudad
        if not os.path.isdir(city_path_lstm_data):
            print(f"  Directorio de ciudad {city_path_lstm_data} no encontrado. Omitiendo {dataset_type_lstm}."); continue

        # Iterar sobre todos los escenarios para esta ciudad y tipo de imputación
        for scenario_folder_name_safe_lstm in os.listdir(city_path_lstm_data):
            scenario_path_lstm_data = os.path.join(city_path_lstm_data, scenario_folder_name_safe_lstm)
            if not os.path.isdir(scenario_path_lstm_data): continue
            
            print(f"    Escenario: {scenario_folder_name_safe_lstm}")
            
            # Iterar sobre las repeticiones (solo 1 para prueba)
            for rep_num_lstm in range(1, N_REPETITIONS_LSTM_PRED_FOR_SCENARIOS + 1):
                file_pattern_for_rep_lstm = f"{city_name_safe_lstm_target}_{scenario_folder_name_safe_lstm}_rep{rep_num_lstm}_*{file_suffix_lstm}"
                repetition_files_lstm = sorted(glob.glob(os.path.join(scenario_path_lstm_data, file_pattern_for_rep_lstm)))
                
                if not repetition_files_lstm:
                    # print(f"      No se encontró archivo para rep {rep_num_lstm} en {scenario_path_lstm_data} con patrón '{file_pattern_for_rep_lstm}'.")
                    continue
                
                rep_file_path_lstm = repetition_files_lstm[0]
                file_name_lstm = os.path.basename(rep_file_path_lstm)
                current_rep_id_lstm = "_".join(file_name_lstm.split('_')[-4:-2]) if "_rep" in file_name_lstm and "_seed" in file_name_lstm else f"rep{rep_num_lstm}"

                try:
                    df_city_data_for_pred_lstm = pd.read_csv(rep_file_path_lstm, usecols=lambda c: c in COLUMNS_TO_LOAD_LSTM_PRED + ['Date', 'City'])
                    if not all(tf in df_city_data_for_pred_lstm.columns for tf in TIME_FEATURES_LSTM_PRED):
                        if 'Date' not in df_city_data_for_pred_lstm.columns: print(f"      'Date' no en {file_name_lstm}."); continue
                        temp_dates_lstm = pd.to_datetime(df_city_data_for_pred_lstm['Date'])
                        for tf_val_lstm in TIME_FEATURES_LSTM_PRED:
                            if tf_val_lstm == 'Hour': df_city_data_for_pred_lstm['Hour'] = temp_dates_lstm.dt.hour
                            elif tf_val_lstm == 'DayOfWeek': df_city_data_for_pred_lstm['DayOfWeek'] = temp_dates_lstm.dt.dayofweek
                            elif tf_val_lstm == 'Month': df_city_data_for_pred_lstm['Month'] = temp_dates_lstm.dt.month
                    df_city_data_for_pred_lstm = df_city_data_for_pred_lstm[df_city_data_for_pred_lstm['City'] == city_name_lstm_target].reset_index(drop=True)
                except Exception as e_load_lstm:
                    print(f"        Error cargando {rep_file_path_lstm}: {e_load_lstm}."); continue
                
                scenario_name_lstm_loop = scenario_folder_name_safe_lstm # Nombre del escenario actual
                rep_id_lstm_loop = current_rep_id_lstm         # ID de la repetición actual

                # Procesar este df_city_data_for_pred_lstm
                if df_city_data_for_pred_lstm is None or df_city_data_for_pred_lstm.empty:
                    print(f"  No hay datos para procesar para {dataset_type_lstm}, Esc: {scenario_name_lstm_loop}, Rep: {rep_id_lstm_loop}. Omitiendo."); continue
                
                # Relleno de NaNs para las features que usará el LSTM
                for col_check_pred_lstm in [TARGET_VARIABLE_LSTM_PRED] + EXOGENOUS_COLS_LSTM_PRED:
                    if col_check_pred_lstm in df_city_data_for_pred_lstm.columns and df_city_data_for_pred_lstm[col_check_pred_lstm].isnull().any():
                        df_city_data_for_pred_lstm[col_check_pred_lstm] = df_city_data_for_pred_lstm[col_check_pred_lstm].ffill().bfill()
                    if col_check_pred_lstm in df_city_data_for_pred_lstm.columns and df_city_data_for_pred_lstm[col_check_pred_lstm].isnull().any():
                        print(f"    No se pudieron rellenar NaNs en {col_check_pred_lstm} para {dataset_type_lstm}, Esc: {scenario_name_lstm_loop}. Omitiendo."); df_city_data_for_pred_lstm = pd.DataFrame(); break
                if df_city_data_for_pred_lstm.empty: continue

                print(f"      Entrenando y evaluando LSTM para Esc: {scenario_name_lstm_loop}, Rep: {rep_id_lstm_loop}")
                horizon_metrics_lstm = train_predict_eval_lstm_multi_horizon(
                    df_city_data_for_pred_lstm, TARGET_VARIABLE_LSTM_PRED, EXOGENOUS_COLS_LSTM_PRED, TIME_FEATURES_LSTM_PRED,
                    N_LAGS_INPUT_LSTM_PRED, PREDICTION_HORIZONS_LSTM_PRED, N_CV_SPLITS_LSTM_PRED, model_params_to_use
                )
                for hm_lstm in horizon_metrics_lstm:
                    all_prediction_metrics_lstm_detailed.append({'Dataset_Type': dataset_type_lstm, 'City': city_name_lstm_target, 'Scenario': scenario_name_lstm_loop, 'Repetition_File_ID': rep_id_lstm_loop, 'Horizon_h': hm_lstm['Horizon'], 'MAE': hm_lstm['MAE'], 'RMSE': hm_lstm['RMSE'], 'MASE': hm_lstm['MASE']})
                if horizon_metrics_lstm and horizon_metrics_lstm[0].get('MAE') is not np.nan:
                    print(f"        Resultados LSTM (H={horizon_metrics_lstm[0]['Horizon']}h): MAE={horizon_metrics_lstm[0]['MAE']:.4f}, RMSE={horizon_metrics_lstm[0]['RMSE']:.4f}, MASE={horizon_metrics_lstm[0]['MASE']:.4f}")
        # Salir del bucle de escenarios después de procesar el primero para los datasets imputados (para prueba rápida)
        # Comenta la siguiente línea para procesar todos los escenarios
        # break 

    # Este bloque es para el caso "Original_Clean" que no tiene bucle de escenarios interno
    if dataset_type_lstm == "Original_Clean" and 'df_city_data_for_pred_lstm' in locals() and not df_city_data_for_pred_lstm.empty:
        print(f"    Entrenando y evaluando LSTM para {dataset_type_lstm}, Escenario: {scenario_name_lstm}, Rep: {rep_id_lstm}")
        horizon_metrics_lstm = train_predict_eval_lstm_multi_horizon(
            df_city_data_for_pred_lstm, TARGET_VARIABLE_LSTM_PRED, EXOGENOUS_COLS_LSTM_PRED, TIME_FEATURES_LSTM_PRED,
            N_LAGS_INPUT_LSTM_PRED, PREDICTION_HORIZONS_LSTM_PRED, N_CV_SPLITS_LSTM_PRED, model_params_to_use
        )
        for hm_lstm in horizon_metrics_lstm:
            all_prediction_metrics_lstm_detailed.append({'Dataset_Type': dataset_type_lstm, 'City': city_name_lstm_target, 'Scenario': scenario_name_lstm, 'Repetition_File_ID': rep_id_lstm, 'Horizon_h': hm_lstm['Horizon'], 'MAE': hm_lstm['MAE'], 'RMSE': hm_lstm['RMSE'], 'MASE': hm_lstm['MASE']})
        if horizon_metrics_lstm and horizon_metrics_lstm[0].get('MAE') is not np.nan:
            print(f"      Resultados LSTM (H={horizon_metrics_lstm[0]['Horizon']}h): MAE={horizon_metrics_lstm[0]['MAE']:.4f}, RMSE={horizon_metrics_lstm[0]['RMSE']:.4f}, MASE={horizon_metrics_lstm[0]['MASE']:.4f}")


# --- Guardado de Métricas ---
df_all_prediction_metrics_lstm_detailed = pd.DataFrame(all_prediction_metrics_lstm_detailed)
if not df_all_prediction_metrics_lstm_detailed.empty:
    detailed_output_path_lstm = os.path.join(PREDICTION_METRICS_LSTM_ROOT_DIR, f'{TARGET_CITY_LSTM_PRED}_prediction_metrics_detailed_{PREDICTION_METHOD_NAME_LSTM_PRED}.csv')
    df_all_prediction_metrics_lstm_detailed.to_csv(detailed_output_path_lstm, index=False)
    print(f"\nMétricas detalladas LSTM ({TARGET_CITY_LSTM_PRED}) guardadas: {detailed_output_path_lstm}")
    if not df_all_prediction_metrics_lstm_detailed.empty: print(df_all_prediction_metrics_lstm_detailed.head())
    city_name_lstm_file_summary = TARGET_CITY_LSTM_PRED.replace(" ", "_").replace("-", "_")
    city_metrics_dir_lstm_summary = os.path.join(PREDICTION_METRICS_LSTM_ROOT_DIR, city_name_lstm_file_summary)
    os.makedirs(city_metrics_dir_lstm_summary, exist_ok=True)
    df_summary_for_target_city_lstm = df_all_prediction_metrics_lstm_detailed[df_all_prediction_metrics_lstm_detailed['City'] == TARGET_CITY_LSTM_PRED]
    if not df_summary_for_target_city_lstm.empty:
        df_summary_by_dataset_scenario_horizon_lstm = df_summary_for_target_city_lstm.groupby(['Dataset_Type', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean'), Count_Repetitions=('MAE', 'size')).reset_index()
        city_summary_file_path_lstm = os.path.join(city_metrics_dir_lstm_summary, f'metrics_summary_{PREDICTION_METHOD_NAME_LSTM_PRED}_{city_name_lstm_file_summary}.csv')
        df_summary_by_dataset_scenario_horizon_lstm.to_csv(city_summary_file_path_lstm, index=False)
        print(f"  Resumen LSTM para {TARGET_CITY_LSTM_PRED} guardado: {city_summary_file_path_lstm}")
    df_global_summary_by_dataset_horizon_lstm = df_all_prediction_metrics_lstm_detailed.groupby(['Dataset_Type', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
    global_summary_file_path_lstm = os.path.join(PREDICTION_METRICS_LSTM_ROOT_DIR, f'{TARGET_CITY_LSTM_PRED}_prediction_metrics_global_summary_{PREDICTION_METHOD_NAME_LSTM_PRED}.csv')
    df_global_summary_by_dataset_horizon_lstm.to_csv(global_summary_file_path_lstm, index=False)
    print(f"\nResumen 'global' LSTM ({TARGET_CITY_LSTM_PRED}) guardado: {global_summary_file_path_lstm}")
    if not df_global_summary_by_dataset_horizon_lstm.empty: print(df_global_summary_by_dataset_horizon_lstm)

script_end_time_lstm_pred = time.time()
print(f"\nScript de predicción LSTM completado. Tiempo: {(script_end_time_lstm_pred - script_start_time_lstm_pred)/60:.2f} min.")

DataFrame original 'c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' cargado para predicción LSTM.

--- Predicción LSTM con Datos: Original_Clean para Brasilia ---
    Entrenando y evaluando LSTM para Original_Clean, Escenario: N/A_Original, Rep: N/A_Original
      Resultados LSTM (H=1h): MAE=4.4762, RMSE=5.8068, MASE=2.8123

--- Predicción LSTM con Datos: Imputed_LinearInterpolation para Brasilia ---
    Escenario: BlockRows_12h
      Entrenando y evaluando LSTM para Esc: BlockRows_12h, Rep: rep1_seed1975935429
        Resultados LSTM (H=1h): MAE=4.9671, RMSE=6.5995, MASE=3.1546
    Escenario: BlockRows_24h
      Entrenando y evaluando LSTM para Esc: BlockRows_24h, Rep: rep1_seed221633945
        Resultados LSTM (H=1h): MAE=4.4924, RMSE=6.1477, MASE=2.8535
    Escenario: BlockRows_2h
      Entrenando y evaluando LSTM para Esc: BlockRows_2h, Rep: rep1_seed2237180940
        Resultados LSTM (H=1h): MAE=5.4752, RMSE=7.0262, MASE=3.4656
    Escenario: Bl